In [1]:
# -*- coding: utf-8 -*-

# Standard library imports
import os
import glob
import warnings
from os.path import join as pjoin

# Data processing and numerical computation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
import seaborn as sns
from tqdm.auto import trange
import xarray as xr
from scipy import stats
from statsmodels.stats.anova import anova_lm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Configure warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Import utility functions from utils.py
from utils import plt_color_dir
# Define utility functions moved from utils.py
def sig_stars(pval, include_ns=False, alpha=0.05):
    """Convert p-value to significance star notation.
    
    Parameters:
    -----------
    pval : float
        P-value to convert
    include_ns : bool, optional
        Whether to return 'n.s.' for non-significant values (default: False)
    alpha : float, optional
        Significance threshold (default: 0.05)
        
    Returns:
    --------
    str
        Significance stars ('*', '**', '***') or empty string/'n.s.' for non-significant
    """
    if pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < alpha:
        return '*'
    else:
        return 'n.s.' if include_ns else ''

def add_sig_plt(ax, x_ind, y_ind, font_size, pval, pos_neg=0):
    """Add significance stars to a plot.
    
    Parameters:
    -----------
    ax : matplotlib.axes.Axes
        Axes to add text to
    x_ind, y_ind : float
        Position coordinates
    font_size : int
        Font size for significance text
    pval : float
        P-value
    pos_neg : int, optional
        0 for black text, 1 for red text
    """
    plt_clr = 'r' if pos_neg == 1 else 'k'
    stars = sig_stars(pval)
    if stars:  # Only add text if there are stars to show
        ax.text(x_ind, y_ind, stars, color=plt_clr, fontsize=font_size, transform=ax.transAxes)

def analyze_tau_values(save_dir, areas_all, n_back):
    """Analyze tau values across brain regions, preserving mouse ID information.
    
    Parameters:
    -----------
    save_dir : str
        Directory where xarray datasets are stored
    areas_all : list
        List of lists containing session names for each brain region
    n_back : int
        Number of trials back for history analysis
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'tau'
    """
    # Define area names based on the structure of areas_all
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    # Create empty list to store all data
    all_data = []
    
    for area_idx in trange(len(areas_all)):
        area_name = area_names[area_idx]
        sessions_all = areas_all[area_idx]
        
        for ss in trange(len(sessions_all)):
            session_name = sessions_all[ss]
            mouse_id = session_name[:5]  # Extract mouse ID (first 5 characters)
            date = session_name[6:12]
            
            print('Checking analysis for {} on {}'.format(mouse_id, date))
            
            # Load xarray dataset
            try:
                output_xarray = xr.open_dataset(pjoin(save_dir, 'cellfits_halves_compare_mdl', 
                                                     'sse_{}_{}hist_updated.nc'.format(session_name, n_back)))
                
                # Apply criteria for cell selection
                crit_RCp1 = output_xarray.sel(mdl_type='Rp1', half=0).p_beta_RC < 0.05
                crit_sig_half1 = output_xarray.sel(mdl_type='exp_r', half=1).p_beta_RC < 0.05
                crit_sig_half2 = output_xarray.sel(mdl_type='exp_r', half=2).p_beta_RC < 0.05
                
                # Extract data to pandas DataFrame
                out_pd = output_xarray.sel(half=0, mdl_type='exp_r').to_dataframe()[['beta_RC', 'tau_RC']]
                out_pd_half1 = output_xarray.sel(half=1, mdl_type='exp_r').to_dataframe()[['tau_RC']]
                out_pd_half2 = output_xarray.sel(half=2, mdl_type='exp_r').to_dataframe()[['tau_RC']]
                
                # Add criteria columns
                out_pd['crit_RCp1'] = crit_RCp1
                out_pd['crit_sig_half1'] = crit_sig_half1
                out_pd['crit_sig_half2'] = crit_sig_half2
                
                # Filter tau values based on criteria
                filtered_taus = out_pd.loc[(out_pd['crit_sig_half1'] == True) & 
                                (out_pd['crit_sig_half2'] == True) & 
                                (out_pd['tau_RC'] < 99.9) & 
                                (out_pd['tau_RC'] >= 0.11), 'tau_RC']
                
                # Add each tau value to the all_data list with metadata
                for tau in filtered_taus:
                    all_data.append({
                        'mouse_id': mouse_id,
                        'area': area_name,
                        'session': session_name,
                        'tau': tau
                    })
                    
            except Exception as e:
                print(f"Error processing session {session_name}: {e}")
    
    # Convert to DataFrame
    tau_df = pd.DataFrame(all_data)
    
    return tau_df

def plot_tau_distribution(save_dir, tau_df, filename_prefix='tau_distribution'):
    """Plot tau distribution across brain regions.
    
    Parameters:
    -----------
    save_dir : str
        Directory to save the figure
    tau_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'tau'
    filename_prefix : str, optional
        Prefix for the saved files (default: 'tau_distribution')
        
    Returns:
    --------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure and axes objects
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Create figure with smaller size
    f, ax = plt.subplots(1, 1, figsize=[3.5, 2.5], tight_layout=False)
    
    # Set up logarithmic bins
    n_bins = 31
    logbins = np.geomspace(10e-2, 10e2, n_bins)
    
    # Extract tau values for each brain region
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    for i, area in enumerate(area_names):
        # Get tau values for this area
        area_taus = tau_df[tau_df['area'] == area]['tau'].values
        
        if len(area_taus) == 0:
            print(f"No tau values found for {area}")
            continue
            
        # Calculate histogram
        Y, _ = np.histogram(area_taus, bins=logbins, density=False)
        Y = Y / len(area_taus)
        X = (np.diff(logbins) / 2) + logbins[:-1]
        
        # Plot differently for first area (RSC) vs others
        if area == 'RSC':
            ax.bar(X, Y, (np.diff(logbins)), facecolor=plt_colors[area], edgecolor='none', 
                  alpha=0.5, label=f'{area} med=' + str(round(np.median(area_taus), 2)))
        else:
            ax.step(X, Y, color=plt_colors[area], where="mid", linewidth=1, 
                   label=f'{area} med=' + str(round(np.median(area_taus), 2)))
    
    # Configure plot appearance
    ax.set_xscale('log')
    ax.set_xlim([10e-2, 10e1])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim([0, .21])
    ax.set_yticks([0.05, 0.1, 0.15, 0.2])
    ax.set_xlabel(r' Cell $\tau$ ', fontsize=12)
    ax.set_ylabel('Fraction of cells', fontsize=12)
    
    # Add legend with smaller font size and better positioning for the smaller figure
    ax.legend(loc='upper right', frameon=False, fontsize=8)
    
    # Adjust subplot positioning
    f.subplots_adjust(left=.22, bottom=.22, right=None, top=None, wspace=None, hspace=None)
    
    # Save figure with the specified prefix
    f.savefig(os.path.join(save_dir, f'{filename_prefix}.png'), dpi=300)
    f.savefig(os.path.join(save_dir, f'{filename_prefix}.svg'))
    
    return f, ax

def plot_tau_boxplot_comparison(save_dir, tau_df, filename_prefix='tau_boxplot_stats'):
    """Create boxplot comparing tau distributions across brain regions with statistical tests.
    
    Parameters:
    -----------
    save_dir : str
        Directory to save the figure
    tau_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'tau'
    filename_prefix : str, optional
        Prefix for the saved files (default: 'tau_boxplot_stats')
        
    Returns:
    --------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure and axes objects
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Extract tau values for each brain region
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    area_taus = {}
    
    for area in area_names:
        area_taus[area] = tau_df[tau_df['area'] == area]['tau'].values
        if len(area_taus[area]) == 0:
            print(f"No tau values found for {area}")
            area_taus[area] = np.array([np.nan])  # Set to NaN to avoid errors
            
    # Create the base figure
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    
    # Setup for plotting boxplots with custom appearance
    positions = [1, 2, 3, 4]  # Position of boxes on x-axis
    box_width = 0.5           # Width of each box
    
    # Set up common box plotting properties
    box_properties = {
        'widths': box_width,
        'showcaps': True,
        'showfliers': False,  # Don't show outliers
        'whiskerprops': {'linewidth': 1, 'linestyle': '-'},
        'boxprops': {'linewidth': 1},
        'medianprops': {'linewidth': 1.5, 'color': 'black'},
        'capprops': {'linewidth': 1}
    }
    
    # Plot boxplots for each area with area-specific color
    for i, area in enumerate(area_names):
        pos = positions[i]
        color = plt_colors[area]
        
        # Create boxplot
        box = ax.boxplot(area_taus[area], positions=[pos], patch_artist=True, **box_properties)
        
        # Set box color
        for patch in box['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    
    # Perform statistical tests comparing RSC to other regions
    # Use non-parametric Mann-Whitney U test
    p_values = {}
    
    for area in ['PPC', 'M2', 'S1']:
        # Mann-Whitney U test
        stat, p = stats.mannwhitneyu(area_taus['RSC'], area_taus[area], alternative='two-sided')
        p_values[f'RSC_vs_{area}'] = p
    
    # Apply FDR correction to p-values
    areas_compared = list(p_values.keys())
    p_vals = [p_values[area] for area in areas_compared]
    reject, p_vals_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
    
    # Update p-values with corrected values
    for i, area in enumerate(areas_compared):
        p_values[area] = p_vals_corrected[i]
    
    # Add significance bars
    y_max = ax.get_ylim()[1]
    bar_height = y_max * 0.08
    text_height = y_max * 0.12
    
    # RSC vs PPC
    if p_values['RSC_vs_PPC'] < 0.05:
        ax.plot([1, 2], [y_max - bar_height, y_max - bar_height], 'k-', linewidth=1)
        ax.text(1.5, y_max - text_height, sig_stars(p_values['RSC_vs_PPC']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs M2
    if p_values['RSC_vs_M2'] < 0.05:
        ax.plot([1, 3], [y_max - 2*bar_height, y_max - 2*bar_height], 'k-', linewidth=1)
        ax.text(2, y_max - 2*text_height, sig_stars(p_values['RSC_vs_M2']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs S1
    if p_values['RSC_vs_S1'] < 0.05:
        ax.plot([1, 4], [y_max - 3*bar_height, y_max - 3*bar_height], 'k-', linewidth=1)
        ax.text(2.5, y_max - 3*text_height, sig_stars(p_values['RSC_vs_S1']), 
                ha='center', va='center', fontsize=9)
    
    # Set axis labels and properties
    ax.set_xticks(positions)
    ax.set_xticklabels(area_names)
    ax.set_ylabel(r'Cell $\tau$', fontsize=12)
    
    # Use log y-scale since tau values span several orders of magnitude
    ax.set_yscale('log')
    
    # Add note about FDR correction
    ax.text(0.5, -0.15, '* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)', 
            ha='center', va='center', transform=ax.transAxes, fontsize=8)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Set constant y-limits
    ax.set_ylim([0.1, 10])
    
    # Adjust layout and save
    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.png'), dpi=300)
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.svg'))
    
    return fig, ax

def plot_tau_barplot_comparison(save_dir, tau_df, filename_prefix='tau_barplot_stats'):
    """Create bar plot comparing session-level tau distributions across brain regions with mean±SEM.
    
    Parameters:
    -----------
    save_dir : str
        Directory to save the figure
    tau_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'tau'
    filename_prefix : str, optional
        Prefix for the saved files (default: 'tau_barplot_stats')
        
    Returns:
    --------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure and axes objects
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Step 1: Calculate median tau value for each session
    session_medians = []
    
    # Process sessions
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = tau_df[tau_df['area'] == area]
        for session in area_data['session'].unique():
            session_data = area_data[area_data['session'] == session]
            mouse_id = session_data['mouse_id'].iloc[0]
            median_tau = session_data['tau'].median()
            session_medians.append({
                'mouse_id': mouse_id,
                'area': area,
                'session': session,
                'median_tau': median_tau
            })
    
    # Convert to DataFrame
    median_df = pd.DataFrame(session_medians)
    
    # Calculate mean and SEM of median_tau for each area
    area_stats = {}
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = median_df[median_df['area'] == area]
        if len(area_data) == 0:
            print(f"No data found for {area}")
            area_stats[area] = {
                'mean': np.nan,
                'sem': np.nan,
                'data': np.array([])
            }
        else:
            area_stats[area] = {
                'mean': np.mean(area_data['median_tau']),
                'sem': stats.sem(area_data['median_tau']),
                'data': area_data['median_tau'].values
            }
    
    # Create the base figure
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    
    # Setup for plotting bars
    positions = [1, 2, 3, 4]  # Position of bars on x-axis
    bar_width = 0.5           # Width of each bar
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    # Plot bars for each area with area-specific color
    means = [area_stats[area]['mean'] for area in area_names]
    sems = [area_stats[area]['sem'] for area in area_names]
    
    bars = ax.bar(positions, means, width=bar_width, yerr=sems, 
                color=[plt_colors[area] for area in area_names], 
                alpha=0.6, edgecolor='black', linewidth=1, 
                error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1})
    
    # Add scatter plots for individual sessions over the bars
    for i, area in enumerate(area_names):
        x_pos = positions[i]
        # Add jitter to x positions for better visibility
        x_jitter = np.random.normal(0, 0.05, size=len(area_stats[area]['data']))
        ax.scatter(x_pos + x_jitter, area_stats[area]['data'], color='black', s=10, alpha=0.5)
    
    # Perform statistical tests comparing RSC to other regions using session-level data
    # Use non-parametric Mann-Whitney U test
    p_values = {}
    
    for area in ['PPC', 'M2', 'S1']:
        if not np.isnan(area_stats['RSC']['mean']) and not np.isnan(area_stats[area]['mean']):
            # Mann-Whitney U test
            stat, p = stats.mannwhitneyu(area_stats['RSC']['data'], area_stats[area]['data'], 
                                        alternative='two-sided')
            p_values[f'RSC_vs_{area}'] = p
        else:
            p_values[f'RSC_vs_{area}'] = 1.0  # No significance if data is missing
    
    # Apply FDR correction to p-values
    areas_compared = list(p_values.keys())
    p_vals = [p_values[area] for area in areas_compared]
    reject, p_vals_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
    
    # Update p-values with corrected values
    for i, area in enumerate(areas_compared):
        p_values[area] = p_vals_corrected[i]
    
    # Add significance bars
    y_max = ax.get_ylim()[1]
    bar_height = y_max * 0.08
    text_height = y_max * 0.12
    
    # RSC vs PPC
    if p_values['RSC_vs_PPC'] < 0.05:
        ax.plot([1, 2], [y_max - bar_height, y_max - bar_height], 'k-', linewidth=1)
        ax.text(1.5, y_max - text_height, sig_stars(p_values['RSC_vs_PPC']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs M2
    if p_values['RSC_vs_M2'] < 0.05:
        ax.plot([1, 3], [y_max - 2*bar_height, y_max - 2*bar_height], 'k-', linewidth=1)
        ax.text(2, y_max - 2*text_height, sig_stars(p_values['RSC_vs_M2']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs S1
    if p_values['RSC_vs_S1'] < 0.05:
        ax.plot([1, 4], [y_max - 3*bar_height, y_max - 3*bar_height], 'k-', linewidth=1)
        ax.text(2.5, y_max - 3*text_height, sig_stars(p_values['RSC_vs_S1']), 
                ha='center', va='center', fontsize=9)
    
    # Set axis labels and properties
    ax.set_xticks(positions)
    ax.set_xticklabels(area_names)
    ax.set_ylabel(r'Session median $\tau$ (mean $\pm$ SEM)', fontsize=12)
    
    # Use log y-scale since tau values span several orders of magnitude
    ax.set_yscale('log')
    
    # Add note about FDR correction
    ax.text(0.5, -0.15, '* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)', 
            ha='center', va='center', transform=ax.transAxes, fontsize=8)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Set constant y-limits (adjust if needed based on data)
    ax.set_ylim([0.1, 10])
    
    # Adjust layout and save
    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.png'), dpi=300)
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.svg'))
    
    return fig, ax

def analyze_beta_values(save_dir, areas_all, n_back):
    """Analyze beta values across brain regions, preserving mouse ID information.
    
    Parameters:
    -----------
    save_dir : str
        Directory where xarray datasets are stored
    areas_all : list
        List of lists containing session names for each brain region
    n_back : int
        Number of trials back for history analysis
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'beta'
    """
    # Define area names based on the structure of areas_all
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    # Create empty list to store all data
    all_data = []
    
    for area_idx in trange(len(areas_all)):
        area_name = area_names[area_idx]
        sessions_all = areas_all[area_idx]
        
        for ss in trange(len(sessions_all)):
            session_name = sessions_all[ss]
            mouse_id = session_name[:5]  # Extract mouse ID (first 5 characters)
            date = session_name[6:12]
            
            print('Checking analysis for {} on {}'.format(mouse_id, date))
            
            # Load xarray dataset
            try:
                output_xarray = xr.open_dataset(pjoin(save_dir, 'cellfits_halves_compare_mdl', 
                                                     'sse_{}_{}hist_updated.nc'.format(session_name, n_back)))
                
                # Apply criteria for cell selection - same as for tau
                crit_RCp1 = output_xarray.sel(mdl_type='Rp1', half=0).p_beta_RC < 0.05
                crit_sig_half1 = output_xarray.sel(mdl_type='exp_r', half=1).p_beta_RC < 0.05
                crit_sig_half2 = output_xarray.sel(mdl_type='exp_r', half=2).p_beta_RC < 0.05
                
                # Extract data to pandas DataFrame
                out_pd = output_xarray.sel(half=0, mdl_type='exp_r').to_dataframe()[['beta_RC', 'tau_RC']]
                
                # Add criteria columns
                out_pd['crit_RCp1'] = crit_RCp1
                out_pd['crit_sig_half1'] = crit_sig_half1
                out_pd['crit_sig_half2'] = crit_sig_half2
                
                # Filter beta values using same tau criteria as for tau_values function
                filtered_betas = out_pd.loc[(out_pd['crit_sig_half1'] == True) & 
                                          (out_pd['crit_sig_half2'] == True) & 
                                          (out_pd['tau_RC'] < 99.9) & 
                                          (out_pd['tau_RC'] >= 0.11), 'beta_RC']
                
                # Add each beta value to the all_data list with metadata
                for beta in filtered_betas:
                    all_data.append({
                        'mouse_id': mouse_id,
                        'area': area_name,
                        'session': session_name,
                        'beta': beta  # Raw beta value (can be negative)
                    })
                    
            except Exception as e:
                print(f"Error processing session {session_name}: {e}")
    
    # Convert to DataFrame
    beta_df = pd.DataFrame(all_data)
    
    return beta_df

def plot_beta_distribution(save_dir, beta_df, filename_prefix='beta_distribution'):
    """Plot beta distribution across brain regions.
    
    Parameters:
    -----------
    save_dir : str
        Directory to save the figure
    beta_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'beta'
    filename_prefix : str, optional
        Prefix for the saved files (default: 'beta_distribution')
        
    Returns:
    --------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure and axes objects
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Create figure with smaller size
    f, ax = plt.subplots(1, 1, figsize=[3.5, 2.5], tight_layout=False)
    
    # Set up linear bins for absolute beta values
    n_bins = 31
    # Use linear bins appropriate for absolute beta value range
    max_abs_beta = np.max(np.abs(beta_df['beta']))
    linear_bins = np.linspace(0, max_abs_beta, n_bins)
    
    # Extract beta values for each brain region
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    for i, area in enumerate(area_names):
        # Get beta values for this area
        area_betas = beta_df[beta_df['area'] == area]['beta'].values
        
        if len(area_betas) == 0:
            print(f"No beta values found for {area}")
            continue
            
        # Take absolute values for plotting
        abs_area_betas = np.abs(area_betas)
        
        # Calculate histogram
        Y, _ = np.histogram(abs_area_betas, bins=linear_bins, density=False)
        Y = Y / len(abs_area_betas)
        X = (np.diff(linear_bins) / 2) + linear_bins[:-1]
        
        # Plot differently for first area (RSC) vs others
        if area == 'RSC':
            ax.bar(X, Y, (np.diff(linear_bins)), facecolor=plt_colors[area], edgecolor='none', 
                  alpha=0.5, label=f'{area} med=' + str(round(np.median(abs_area_betas), 2)))
        else:
            ax.step(X, Y, color=plt_colors[area], where="mid", linewidth=1, 
                   label=f'{area} med=' + str(round(np.median(abs_area_betas), 2)))
    
    # Configure plot appearance
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim([0, .3])
    ax.set_xlim([0, .6])
    ax.set_yticks([0.05, 0.1, 0.15, 0.2, 0.25, 0.3])
    ax.set_xticks([0, .2, .4, .6])
    ax.set_xlabel(r'|Cell $\beta$|', fontsize=12)
    ax.set_ylabel('Fraction of cells', fontsize=12)
    
    # Add legend with smaller font size and better positioning for the smaller figure
    ax.legend(loc='upper right', frameon=False, fontsize=8)
    
    # Adjust subplot positioning
    f.subplots_adjust(left=.22, bottom=.22, right=None, top=None, wspace=None, hspace=None)
    
    # Save figure with the specified prefix
    f.savefig(os.path.join(save_dir, f'{filename_prefix}.png'), dpi=300)
    f.savefig(os.path.join(save_dir, f'{filename_prefix}.svg'))
    
    return f, ax

def plot_beta_boxplot_comparison(save_dir, beta_df, filename_prefix='beta_boxplot_stats'):
    """Create boxplot comparing beta distributions across brain regions with statistical tests.
    
    Parameters:
    -----------
    save_dir : str
        Directory to save the figure
    beta_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'beta'
    filename_prefix : str, optional
        Prefix for the saved files (default: 'beta_boxplot_stats')
        
    Returns:
    --------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure and axes objects
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Create a new column for absolute beta values
    beta_df = beta_df.copy()
    beta_df['abs_beta'] = np.abs(beta_df['beta'])
    
    # Extract abs beta values for each brain region
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    area_betas = {}
    
    for area in area_names:
        area_betas[area] = beta_df[beta_df['area'] == area]['abs_beta'].values
        if len(area_betas[area]) == 0:
            print(f"No beta values found for {area}")
            area_betas[area] = np.array([np.nan])  # Set to NaN to avoid errors
            
    # Create the base figure
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    
    # Setup for plotting boxplots with custom appearance
    positions = [1, 2, 3, 4]  # Position of boxes on x-axis
    box_width = 0.5           # Width of each box
    
    # Set up common box plotting properties
    box_properties = {
        'widths': box_width,
        'showcaps': True,
        'showfliers': False,  # Don't show outliers
        'whiskerprops': {'linewidth': 1, 'linestyle': '-'},
        'boxprops': {'linewidth': 1},
        'medianprops': {'linewidth': 1.5, 'color': 'black'},
        'capprops': {'linewidth': 1}
    }
    
    # Plot boxplots for each area with area-specific color
    for i, area in enumerate(area_names):
        pos = positions[i]
        color = plt_colors[area]
        
        # Create boxplot
        box = ax.boxplot(area_betas[area], positions=[pos], patch_artist=True, **box_properties)
        
        # Set box color
        for patch in box['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    
    # Perform statistical tests comparing RSC to other regions
    # Use non-parametric Mann-Whitney U test
    p_values = {}
    
    for area in ['PPC', 'M2', 'S1']:
        # Mann-Whitney U test
        stat, p = stats.mannwhitneyu(area_betas['RSC'], area_betas[area], alternative='two-sided')
        p_values[f'RSC_vs_{area}'] = p
    
    # Apply FDR correction to p-values
    areas_compared = list(p_values.keys())
    p_vals = [p_values[area] for area in areas_compared]
    reject, p_vals_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
    
    # Update p-values with corrected values
    for i, area in enumerate(areas_compared):
        p_values[area] = p_vals_corrected[i]
    
    # Add significance bars
    y_max = ax.get_ylim()[1]
    bar_height = y_max * 0.08
    text_height = y_max * 0.12
    
    # RSC vs PPC
    if p_values['RSC_vs_PPC'] < 0.05:
        ax.plot([1, 2], [y_max - bar_height, y_max - bar_height], 'k-', linewidth=1)
        ax.text(1.5, y_max - text_height, sig_stars(p_values['RSC_vs_PPC']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs M2
    if p_values['RSC_vs_M2'] < 0.05:
        ax.plot([1, 3], [y_max - 2*bar_height, y_max - 2*bar_height], 'k-', linewidth=1)
        ax.text(2, y_max - 2*text_height, sig_stars(p_values['RSC_vs_M2']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs S1
    if p_values['RSC_vs_S1'] < 0.05:
        ax.plot([1, 4], [y_max - 3*bar_height, y_max - 3*bar_height], 'k-', linewidth=1)
        ax.text(2.5, y_max - 3*text_height, sig_stars(p_values['RSC_vs_S1']), 
                ha='center', va='center', fontsize=9)
    
    # Set axis labels and properties
    ax.set_xticks(positions)
    ax.set_xticklabels(area_names)
    ax.set_ylabel(r'|Cell $\beta$|', fontsize=12)
    
    # Use linear y-scale for beta values
    ax.set_ylim([0, 0.6])
    
    # Add note about FDR correction
    ax.text(0.5, -0.15, '* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)', 
            ha='center', va='center', transform=ax.transAxes, fontsize=8)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Adjust layout and save
    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.png'), dpi=300)
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.svg'))
    
    return fig, ax

def plot_beta_barplot_comparison(save_dir, beta_df, filename_prefix='beta_barplot_stats'):
    """Create bar plot comparing session-level beta distributions across brain regions with mean±SEM.
    
    Parameters:
    -----------
    save_dir : str
        Directory to save the figure
    beta_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'session', 'beta'
    filename_prefix : str, optional
        Prefix for the saved files (default: 'beta_barplot_stats')
        
    Returns:
    --------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        Figure and axes objects
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Create a new column for absolute beta values
    beta_df = beta_df.copy()
    beta_df['abs_beta'] = np.abs(beta_df['beta'])
    
    # Step 1: Calculate median abs_beta value for each session
    session_medians = []
    
    # Process sessions
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = beta_df[beta_df['area'] == area]
        for session in area_data['session'].unique():
            session_data = area_data[area_data['session'] == session]
            mouse_id = session_data['mouse_id'].iloc[0]
            median_beta = session_data['abs_beta'].median()
            session_medians.append({
                'mouse_id': mouse_id,
                'area': area,
                'session': session,
                'median_beta': median_beta
            })
    
    # Convert to DataFrame
    median_df = pd.DataFrame(session_medians)
    
    # Calculate mean and SEM of median_beta for each area
    area_stats = {}
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = median_df[median_df['area'] == area]
        if len(area_data) == 0:
            print(f"No data found for {area}")
            area_stats[area] = {
                'mean': np.nan,
                'sem': np.nan,
                'data': np.array([])
            }
        else:
            area_stats[area] = {
                'mean': np.mean(area_data['median_beta']),
                'sem': stats.sem(area_data['median_beta']),
                'data': area_data['median_beta'].values
            }
    
    # Create the base figure
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    
    # Setup for plotting bars
    positions = [1, 2, 3, 4]  # Position of bars on x-axis
    bar_width = 0.5           # Width of each bar
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    # Plot bars for each area with area-specific color
    means = [area_stats[area]['mean'] for area in area_names]
    sems = [area_stats[area]['sem'] for area in area_names]
    
    bars = ax.bar(positions, means, width=bar_width, yerr=sems, 
                color=[plt_colors[area] for area in area_names], 
                alpha=0.6, edgecolor='black', linewidth=1, 
                error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1})
    
    # Add scatter plots for individual sessions over the bars
    for i, area in enumerate(area_names):
        x_pos = positions[i]
        # Add jitter to x positions for better visibility
        x_jitter = np.random.normal(0, 0.05, size=len(area_stats[area]['data']))
        ax.scatter(x_pos + x_jitter, area_stats[area]['data'], color='black', s=10, alpha=0.5)
    
    # Perform statistical tests comparing RSC to other regions using session-level data
    # Use non-parametric Mann-Whitney U test
    p_values = {}
    
    for area in ['PPC', 'M2', 'S1']:
        if not np.isnan(area_stats['RSC']['mean']) and not np.isnan(area_stats[area]['mean']):
            # Mann-Whitney U test
            stat, p = stats.mannwhitneyu(area_stats['RSC']['data'], area_stats[area]['data'], 
                                        alternative='two-sided')
            p_values[f'RSC_vs_{area}'] = p
        else:
            p_values[f'RSC_vs_{area}'] = 1.0  # No significance if data is missing
    
    # Apply FDR correction to p-values
    areas_compared = list(p_values.keys())
    p_vals = [p_values[area] for area in areas_compared]
    reject, p_vals_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
    
    # Update p-values with corrected values
    for i, area in enumerate(areas_compared):
        p_values[area] = p_vals_corrected[i]
    
    # Add significance bars
    y_max = 0.2  # Fixed max value for beta plots
    bar_height = y_max * 0.08
    text_height = y_max * 0.12
    
    # RSC vs PPC
    if p_values['RSC_vs_PPC'] < 0.05:
        ax.plot([1, 2], [y_max - bar_height, y_max - bar_height], 'k-', linewidth=1)
        ax.text(1.5, y_max - text_height, sig_stars(p_values['RSC_vs_PPC']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs M2
    if p_values['RSC_vs_M2'] < 0.05:
        ax.plot([1, 3], [y_max - 2*bar_height, y_max - 2*bar_height], 'k-', linewidth=1)
        ax.text(2, y_max - 2*text_height, sig_stars(p_values['RSC_vs_M2']), 
                ha='center', va='center', fontsize=9)
    
    # RSC vs S1
    if p_values['RSC_vs_S1'] < 0.05:
        ax.plot([1, 4], [y_max - 3*bar_height, y_max - 3*bar_height], 'k-', linewidth=1)
        ax.text(2.5, y_max - 3*text_height, sig_stars(p_values['RSC_vs_S1']), 
                ha='center', va='center', fontsize=9)
    
    # Set axis labels and properties
    ax.set_xticks(positions)
    ax.set_xticklabels(area_names)
    ax.set_ylabel(r'Session median |$\beta$| (mean $\pm$ SEM)', fontsize=12)
    
    # Use linear scale for beta values with fixed limits
    ax.set_ylim([0, 0.2])
    
    # Add note about FDR correction
    ax.text(0.5, -0.15, '* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)', 
            ha='center', va='center', transform=ax.transAxes, fontsize=8)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Adjust layout and save
    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.png'), dpi=300)
    fig.savefig(os.path.join(save_dir, f'{filename_prefix}.svg'))
    
    return fig, ax

def analyze_region_adaptation(df_short, df_long, save_dir, fig_number=5):
    """
    Analyze how different brain regions adapt tau values between short and long task conditions.
    Uses mixed effect models with task condition as fixed effect and mouse as random intercept.
    
    Parameters:
    -----------
    df_short, df_long : pandas.DataFrame
        DataFrames with tau values for short and long task conditions
    save_dir : str
        Directory to save the output figures
    fig_number : int, optional
        Figure number to use in filenames (default: 5)
        
    Returns:
    --------
    dict
        Results containing model summaries and statistical comparisons
    """
    # Step 1: Calculate median tau value for each session
    session_medians = []
    plt_colors = plt_color_dir()
    
    # Process short task sessions
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = df_short[df_short['area'] == area]
        for session in area_data['session'].unique():
            session_data = area_data[area_data['session'] == session]
            mouse_id = session_data['mouse_id'].iloc[0]
            median_tau = session_data['tau'].median()
            session_medians.append({
                'mouse_id': mouse_id,
                'area': area,
                'condition': 'short',
                'session': session,
                'median_tau': median_tau
            })
    
    # Process long task sessions
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = df_long[df_long['area'] == area]
        for session in area_data['session'].unique():
            session_data = area_data[area_data['session'] == session]
            mouse_id = session_data['mouse_id'].iloc[0]
            median_tau = session_data['tau'].median()
            session_medians.append({
                'mouse_id': mouse_id,
                'area': area,
                'condition': 'long',
                'session': session,
                'median_tau': median_tau
            })
    
    # Convert to DataFrame
    median_df = pd.DataFrame(session_medians)
    
    # Create a composite group column for plotting
    median_df['group'] = median_df['area'] + '_' + median_df['condition']
    
    # Print summary statistics
    print("Number of sessions per group:")
    print(median_df.groupby(['area', 'condition']).size())
    print("\nNumber of mice per group:")
    print(median_df.groupby(['area', 'condition'])['mouse_id'].nunique())
    print("\nMedian tau values per group:")
    print(median_df.groupby(['area', 'condition'])['median_tau'].median())
    
    # Initialize results dictionary
    results = {}
    
    #######################################################################
    # Mixed Effects Model for each brain region separately
    #######################################################################
    
    mixed_effect_results = {}
    
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = median_df[median_df['area'] == area]
        
        try:
            # Mixed effects model with condition as fixed effect and mouse as random intercept
            model = smf.mixedlm(
                "median_tau ~ condition", 
                area_data, 
                groups=area_data["mouse_id"]
            )
            model_fit = model.fit(reml=True)
            
            # Extract p-value for condition effect
            param_names = list(model_fit.pvalues.index)
            condition_params = [p for p in param_names if 'condition' in p.lower()]
            
            if condition_params:
                p_val = model_fit.pvalues[condition_params[0]]
                effect_size = model_fit.params[condition_params[0]]
                mixed_effect_results[area] = {
                    'summary': model_fit.summary(),
                    'p_value': p_val,
                    'effect_size': effect_size,
                    'converged': True
                }
                print(f"{area} - Mixed effects p-value: {p_val:.4f}, effect size: {effect_size:.4f}")
            else:
                mixed_effect_results[area] = {
                    'converged': False, 
                    'error': 'No condition parameter found'
                }
                print(f"{area} - Mixed effects: No condition parameter found")
        except Exception as e:
            mixed_effect_results[area] = {'converged': False, 'error': str(e)}
            print(f"{area} - Mixed effects model failed: {e}")
    
    # Store results
    results['mixed_effects'] = mixed_effect_results
    
    # Apply FDR correction to p-values from mixed effects models
    print("\nApplying FDR correction to mixed effects p-values:")
    pvals = []
    areas = []
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        if area in mixed_effect_results and 'p_value' in mixed_effect_results[area]:
            pvals.append(mixed_effect_results[area]['p_value'])
            areas.append(area)
    
    if pvals:
        # Use multipletests from our imports at the top level 
        _, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')
        
        # Update results with corrected p-values
        for i, area in enumerate(areas):
            mixed_effect_results[area]['p_value_raw'] = mixed_effect_results[area]['p_value']
            mixed_effect_results[area]['p_value'] = pvals_corrected[i]
            
        # Print a clear comparison of raw and FDR-corrected p-values
        print("\n=== Mixed Effects Model Results with FDR Correction ===")
        print("Region | Raw p-value | FDR-corrected p-value | Effect Size | Significance")
        print("----------------------------------------------------------------------")
        for area in ['RSC', 'PPC', 'M2', 'S1']:
            if area in mixed_effect_results and 'p_value_raw' in mixed_effect_results[area]:
                raw_p = mixed_effect_results[area]['p_value_raw']
                fdr_p = mixed_effect_results[area]['p_value']
                effect = mixed_effect_results[area]['effect_size']
                stars = sig_stars(fdr_p)
                print(f"{area:<5} | {raw_p:.6f} | {fdr_p:.6f} | {effect:.4f} | {stars}")
    
    ###################################################################
    # Generate Plot
    ###################################################################
    
    # Plot for mixed effects within-area results using boxplots
    plt.figure(figsize=(10, 6))
    
    # Define custom order of groups
    order = ['RSC_short', 'RSC_long', 'PPC_short', 'PPC_long', 
             'M2_short', 'M2_long', 'S1_short', 'S1_long']
    
    # Define custom colors
    colors = {}
    for group in order:
        area = group.split('_')[0]  # Extract area (RSC, PPC, etc.)
        condition = group.split('_')[1]  # Extract condition (short or long)
        
        # Use the area's color from plt_colors
        base_color = plt_colors[area]
        
        # For short condition, make the color lighter
        if condition == 'short':
            # Safely create a lighter version
            light_color = [min(c + 0.3, 1.0) for c in base_color]
            colors[group] = light_color
        else:
            colors[group] = base_color
    
    # Create boxplot with custom colors
    ax = sns.boxplot(x='group', y='median_tau', data=median_df, order=order, 
                    palette=colors, showfliers=False)
    
    # Add individual data points
    sns.stripplot(x='group', y='median_tau', data=median_df, order=order,
                 jitter=True, dodge=True, color='black', alpha=0.5, size=4)
    
    # Add vertical lines between regions
    for i in [1.5, 3.5, 5.5]:
        plt.axvline(x=i, color='gray', linestyle='--', alpha=0.5)
    
    # Customize plot
    plt.title(f'Figure {fig_number}: Tau Analysis - Mixed Effects Model by Region', fontsize=14)
    plt.xlabel('Brain Region and Task Condition', fontsize=12)
    plt.ylabel('Median τ', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    # Set y-axis to linear scale with limits [0, 5]
    plt.ylim([0, 5])
    
    # Add a note about FDR correction
    plt.figtext(0.5, 0.01, "* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)", 
               ha='center', fontsize=10, bbox=dict(facecolor='lightyellow', alpha=0.5))
    
    # Add significance bars for mixed effects results
    # Use fixed y position for all significance bars
    y_bar = 4.5
    y_text = 4.7  # Position for the asterisks (slightly above the bar)
    
    bar_positions = {
        'RSC': (0, 1),  # RSC short to RSC long
        'PPC': (2, 3),  # PPC short to PPC long
        'M2': (4, 5),   # M2 short to M2 long
        'S1': (6, 7)    # S1 short to S1 long
    }
    
    for area, (pos1, pos2) in bar_positions.items():
        if area in mixed_effect_results and 'p_value' in mixed_effect_results[area]:
            p_val = mixed_effect_results[area]['p_value']
            stars = sig_stars(p_val)
            
            if stars:  # Only add bars for significant comparisons
                plt.plot([pos1, pos2], [y_bar, y_bar], '-k', linewidth=1)
                mid_pos = (pos1 + pos2) / 2
                plt.text(mid_pos, y_text, stars, ha='center', fontsize=12)
    
    # Remove top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    
    # Adjust layout and save
    plt.tight_layout()
    plt.savefig(f"{save_dir}/Figure{fig_number}_tau_mixed_effects.png", dpi=300)
    plt.savefig(f"{save_dir}/Figure{fig_number}_tau_mixed_effects.svg")
    plt.close()
    
    return results, median_df

def plot_tau_adaptation_barplot(median_df, save_dir, fig_number=5, mixed_effects_results=None):
    """
    Create a barplot comparing tau values between short and long task conditions across brain regions.
    
    Parameters:
    -----------
    median_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'condition', 'session', 'median_tau', 'group'
    save_dir : str
        Directory to save the figure
    fig_number : int, optional
        Figure number for file naming (default: 5)
    mixed_effects_results : dict, optional
        Results from analyze_region_adaptation's mixed effects models to use for significance
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Calculate mean and SEM for each group
    group_stats = {}
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        for condition in ['short', 'long']:
            group = f"{area}_{condition}"
            group_data = median_df[median_df['group'] == group]
            
            # Calculate stats
            if len(group_data) > 0:
                mean_val = group_data['median_tau'].mean()
                sem_val = stats.sem(group_data['median_tau'].values)
                group_stats[group] = {
                    'mean': mean_val,
                    'sem': sem_val,
                    'data': group_data['median_tau'].values
                }
            else:
                group_stats[group] = {
                    'mean': np.nan,
                    'sem': np.nan,
                    'data': np.array([])
                }
    
    # Setup bar positions
    bar_width = 0.35
    index = np.arange(4)  # 4 brain regions
    
    # Create bars
    short_bars = plt.bar(index - bar_width/2, 
                         [group_stats[f'{area}_short']['mean'] for area in ['RSC', 'PPC', 'M2', 'S1']], 
                         bar_width,
                         yerr=[group_stats[f'{area}_short']['sem'] for area in ['RSC', 'PPC', 'M2', 'S1']],
                         color=[plt_colors[area] for area in ['RSC', 'PPC', 'M2', 'S1']],
                         alpha=0.7,
                         edgecolor='black',
                         linewidth=1,
                         error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1},
                         label='Short Task')
    
    long_bars = plt.bar(index + bar_width/2, 
                        [group_stats[f'{area}_long']['mean'] for area in ['RSC', 'PPC', 'M2', 'S1']], 
                        bar_width,
                        yerr=[group_stats[f'{area}_long']['sem'] for area in ['RSC', 'PPC', 'M2', 'S1']],
                        color=[plt_colors[area] for area in ['RSC', 'PPC', 'M2', 'S1']],
                        alpha=0.3,
                        edgecolor='black',
                        linewidth=1,
                        error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1},
                        label='Long Task')
    
    # Add scatter points for individual sessions
    for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
        # Short condition
        short_data = group_stats[f'{area}_short']['data']
        if len(short_data) > 0:
            short_jitter = np.random.normal(0, 0.03, size=len(short_data))
            plt.scatter(i - bar_width/2 + short_jitter, short_data, color='black', s=15, alpha=0.6)
        
        # Long condition
        long_data = group_stats[f'{area}_long']['data']
        if len(long_data) > 0:
            long_jitter = np.random.normal(0, 0.03, size=len(long_data))
            plt.scatter(i + bar_width/2 + long_jitter, long_data, color='black', s=15, alpha=0.6)
    
    # Add significance markers
    # If mixed_effects_results is provided, use those p-values
    # Otherwise, calculate Mann-Whitney tests
    if mixed_effects_results and 'mixed_effects' in mixed_effects_results:
        # Using the same p-values as in the mixed effects model
        p_values = {}
        mixed_results = mixed_effects_results['mixed_effects']
        
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            if area in mixed_results and 'p_value' in mixed_results[area]:
                p_values[area] = mixed_results[area]['p_value']
            else:
                p_values[area] = 1.0  # No significance if data is missing
                
        # Add significance markers based on mixed effects p-values
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            if area in p_values and p_values[area] < 0.05:
                # Add line connecting the bars
                bar_height = 4.5  # Fixed position for significance bars
                plt.plot([i - bar_width/2, i + bar_width/2], [bar_height, bar_height], 'k-', linewidth=1)
                
                # Add stars
                plt.text(i, 4.7, sig_stars(p_values[area]), ha='center', fontsize=12)
                
        # Add note about statistical approach
        subtitle = "Statistical significance based on mixed effects models (same as Figure5_tau_mixed_effects)"
        plt.figtext(0.5, 0.94, subtitle, ha='center', fontsize=10, fontweight='normal')
    else:
        # Fallback to Mann-Whitney tests if no mixed effects results provided
        p_values = {}
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            # Get short and long data
            short_data = group_stats[f'{area}_short']['data']
            long_data = group_stats[f'{area}_long']['data']
            
            if len(short_data) > 0 and len(long_data) > 0:
                # Perform Mann-Whitney test
                stat, p_val = stats.mannwhitneyu(short_data, long_data, alternative='two-sided')
                p_values[area] = p_val
            else:
                p_values[area] = 1.0  # No significance if data is missing
        
        # Apply FDR correction to p-values
        areas = list(p_values.keys())
        p_vals = [p_values[area] for area in areas]
        if p_vals:
            reject, p_vals_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
            
            # Update p-values with corrected values
            for i, area in enumerate(areas):
                p_values[area] = p_vals_corrected[i]
        
        # Add significance markers based on corrected p-values
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            if area in p_values and p_values[area] < 0.05:
                # Add line connecting the bars
                bar_height = 4.5  # Fixed position for significance bars
                plt.plot([i - bar_width/2, i + bar_width/2], [bar_height, bar_height], 'k-', linewidth=1)
                
                # Add stars
                plt.text(i, 4.7, sig_stars(p_values[area]), ha='center', fontsize=12)
                
        # Add note about statistical approach
        subtitle = "Statistical significance based on Mann-Whitney tests"
        plt.figtext(0.5, 0.94, subtitle, ha='center', fontsize=10, fontweight='normal')
    
    # Customize plot
    plt.xlabel('Brain Region', fontsize=14)
    plt.ylabel('Median τ value', fontsize=14)
    plt.title(f'Figure {fig_number}: Tau Adaptation by Region (Short vs Long Task)', fontsize=16)
    plt.xticks(index, ['RSC', 'PPC', 'M2', 'S1'], fontsize=12)
    plt.legend(fontsize=12)
    
    # Use linear scale with limits [0, 5]
    plt.ylim([0, 5])
    
    # Remove top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    
    # Add note about statistical significance
    plt.figtext(0.5, 0.01, "* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)", 
               ha='center', fontsize=10, bbox=dict(facecolor='lightyellow', alpha=0.5))
    
    # Save figure
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'Figure{fig_number}_tau_adaptation_barplot.png'), dpi=300)
    plt.savefig(os.path.join(save_dir, f'Figure{fig_number}_tau_adaptation_barplot.svg'))
    plt.close()
    
    print(f"Figure {fig_number} tau adaptation barplot saved.")

def analyze_beta_region_adaptation(beta_df_short, beta_df_long, save_dir, fig_number=6):
    """
    Analyze how different brain regions adapt beta values between short and long task conditions.
    Uses mixed effect models with task condition as fixed effect and mouse as random intercept.
    
    Parameters:
    -----------
    beta_df_short, beta_df_long : pandas.DataFrame
        DataFrames with beta values for short and long task conditions
    save_dir : str
        Directory to save the output figures
    fig_number : int, optional
        Figure number to use in filenames (default: 6)
        
    Returns:
    --------
    dict
        Results containing model summaries and statistical comparisons
    """
    # Get consistent color scheme
    plt_colors = plt_color_dir()
    
    # Step 1: Calculate median beta value for each session (using absolute values)
    session_medians = []
    
    # Process short task sessions
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = beta_df_short[beta_df_short['area'] == area]
        for session in area_data['session'].unique():
            session_data = area_data[area_data['session'] == session]
            mouse_id = session_data['mouse_id'].iloc[0]
            # Use absolute values for beta
            abs_betas = np.abs(session_data['beta'])
            median_beta = abs_betas.median()
            session_medians.append({
                'mouse_id': mouse_id,
                'area': area,
                'condition': 'short',
                'session': session,
                'median_beta': median_beta
            })
    
    # Process long task sessions
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = beta_df_long[beta_df_long['area'] == area]
        for session in area_data['session'].unique():
            session_data = area_data[area_data['session'] == session]
            mouse_id = session_data['mouse_id'].iloc[0]
            # Use absolute values for beta
            abs_betas = np.abs(session_data['beta'])
            median_beta = abs_betas.median()
            session_medians.append({
                'mouse_id': mouse_id,
                'area': area,
                'condition': 'long',
                'session': session,
                'median_beta': median_beta
            })
    
    # Convert to DataFrame
    median_df = pd.DataFrame(session_medians)
    
    # Create a composite group column for plotting
    median_df['group'] = median_df['area'] + '_' + median_df['condition']
    
    # Print summary statistics
    print("Number of sessions per group:")
    print(median_df.groupby(['area', 'condition']).size())
    print("\nNumber of mice per group:")
    print(median_df.groupby(['area', 'condition'])['mouse_id'].nunique())
    print("\nMedian beta values per group:")
    print(median_df.groupby(['area', 'condition'])['median_beta'].median())
    
    # Initialize results dictionary
    results = {}
    
    #######################################################################
    # Mixed Effects Model for each brain region separately
    #######################################################################
    
    mixed_effect_results = {}
    
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = median_df[median_df['area'] == area]
        
        try:
            # Mixed effects model with condition as fixed effect and mouse as random intercept
            model = smf.mixedlm(
                "median_beta ~ condition", 
                area_data, 
                groups=area_data["mouse_id"]
            )
            model_fit = model.fit(reml=True)
            
            # Extract p-value for condition effect
            param_names = list(model_fit.pvalues.index)
            condition_params = [p for p in param_names if 'condition' in p.lower()]
            
            if condition_params:
                p_val = model_fit.pvalues[condition_params[0]]
                effect_size = model_fit.params[condition_params[0]]
                mixed_effect_results[area] = {
                    'summary': model_fit.summary(),
                    'p_value': p_val,
                    'effect_size': effect_size,
                    'converged': True
                }
                print(f"{area} - Mixed effects p-value: {p_val:.4f}, effect size: {effect_size:.4f}")
            else:
                mixed_effect_results[area] = {
                    'converged': False, 
                    'error': 'No condition parameter found'
                }
                print(f"{area} - Mixed effects: No condition parameter found")
        except Exception as e:
            mixed_effect_results[area] = {'converged': False, 'error': str(e)}
            print(f"{area} - Mixed effects model failed: {e}")
    
    # Store results
    results['mixed_effects'] = mixed_effect_results
    
    # Apply FDR correction to p-values from mixed effects models
    print("\nApplying FDR correction to mixed effects p-values:")
    pvals = []
    areas = []
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        if area in mixed_effect_results and 'p_value' in mixed_effect_results[area]:
            pvals.append(mixed_effect_results[area]['p_value'])
            areas.append(area)
    
    if pvals:
        # Use multipletests from our imports at the top level 
        _, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')
        
        # Update results with corrected p-values
        for i, area in enumerate(areas):
            mixed_effect_results[area]['p_value_raw'] = mixed_effect_results[area]['p_value']
            mixed_effect_results[area]['p_value'] = pvals_corrected[i]
            
        # Print a clear comparison of raw and FDR-corrected p-values
        print("\n=== Mixed Effects Model Results with FDR Correction ===")
        print("Region | Raw p-value | FDR-corrected p-value | Effect Size | Significance")
        print("----------------------------------------------------------------------")
        for area in ['RSC', 'PPC', 'M2', 'S1']:
            if area in mixed_effect_results and 'p_value_raw' in mixed_effect_results[area]:
                raw_p = mixed_effect_results[area]['p_value_raw']
                fdr_p = mixed_effect_results[area]['p_value']
                effect = mixed_effect_results[area]['effect_size']
                stars = sig_stars(fdr_p)
                print(f"{area:<5} | {raw_p:.6f} | {fdr_p:.6f} | {effect:.4f} | {stars}")
    
    ###################################################################
    # Generate Plot
    ###################################################################
    
    # Plot for mixed effects within-area results using boxplots
    plt.figure(figsize=(10, 6))
    
    # Define custom order of groups
    order = ['RSC_short', 'RSC_long', 'PPC_short', 'PPC_long', 
             'M2_short', 'M2_long', 'S1_short', 'S1_long']
    
    # Define custom colors
    colors = {}
    for group in order:
        area = group.split('_')[0]  # Extract area (RSC, PPC, etc.)
        condition = group.split('_')[1]  # Extract condition (short or long)
        
        # Use the area's color from plt_colors
        base_color = plt_colors[area]
        
        # For short condition, make the color lighter
        if condition == 'short':
            # Safely create a lighter version
            light_color = [min(c + 0.3, 1.0) for c in base_color]
            colors[group] = light_color
        else:
            colors[group] = base_color
    
    # Create boxplot with custom colors
    ax = sns.boxplot(x='group', y='median_beta', data=median_df, order=order, 
                    palette=colors, showfliers=False)
    
    # Add individual data points
    sns.stripplot(x='group', y='median_beta', data=median_df, order=order,
                 jitter=True, dodge=True, color='black', alpha=0.5, size=4)
    
    # Add vertical lines between regions
    for i in [1.5, 3.5, 5.5]:
        plt.axvline(x=i, color='gray', linestyle='--', alpha=0.5)
    
    # Customize plot
    plt.title(f'Figure {fig_number}: Beta Analysis - Mixed Effects Model by Region', fontsize=14)
    plt.xlabel('Brain Region and Task Condition', fontsize=12)
    plt.ylabel('Median |β|', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    # Set y-axis limits
    plt.ylim([0, 0.2])
    
    # Add a note about FDR correction
    plt.figtext(0.5, 0.01, "* p<0.05, ** p<0.01, *** p<0.001", 
               ha='center', fontsize=10, bbox=dict(facecolor='lightyellow', alpha=0.5))
    
    # Add significance bars for mixed effects results
    # Use fixed y position for all significance bars
    y_bar = 0.15
    y_text = 0.157  # Position for the asterisks (slightly above the bar)
    
    bar_positions = {
        'RSC': (0, 1),  # RSC short to RSC long
        'PPC': (2, 3),  # PPC short to PPC long
        'M2': (4, 5),   # M2 short to M2 long
        'S1': (6, 7)    # S1 short to S1 long
    }
    
    for area, (pos1, pos2) in bar_positions.items():
        if area in mixed_effect_results and 'p_value' in mixed_effect_results[area]:
            p_val = mixed_effect_results[area]['p_value']
            stars = sig_stars(p_val)
            
            if stars:  # Only add bars for significant comparisons
                plt.plot([pos1, pos2], [y_bar, y_bar], '-k', linewidth=1)
                mid_pos = (pos1 + pos2) / 2
                plt.text(mid_pos, y_text, stars, ha='center', fontsize=12)
    
    # Set final y-axis limits and ticks (overrides previous settings)
    plt.ylim([0, 0.2])
    
    # Remove top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    
    # Adjust layout and save
    plt.tight_layout()
    plt.savefig(f"{save_dir}/Figure{fig_number}_beta_mixed_effects.png", dpi=300)
    plt.savefig(f"{save_dir}/Figure{fig_number}_beta_mixed_effects.svg")
    plt.close()
    
    return results, median_df

def plot_beta_adaptation_barplot(median_df, save_dir, fig_number=6, mixed_effects_results=None):
    """
    Create a barplot comparing beta values between short and long task conditions across brain regions.
    
    Parameters:
    -----------
    median_df : pandas.DataFrame
        DataFrame with columns: 'mouse_id', 'area', 'condition', 'session', 'median_beta', 'group'
    save_dir : str
        Directory to save the figure
    fig_number : int, optional
        Figure number for file naming (default: 6)
    mixed_effects_results : dict, optional
        Results from analyze_beta_region_adaptation's mixed effects models to use for significance
    """
    # Get color scheme
    plt_colors = plt_color_dir()
    
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Calculate mean and SEM for each group
    group_stats = {}
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        for condition in ['short', 'long']:
            group = f"{area}_{condition}"
            group_data = median_df[median_df['group'] == group]
            
            # Calculate stats
            if len(group_data) > 0:
                mean_val = group_data['median_beta'].mean()
                sem_val = stats.sem(group_data['median_beta'].values)
                group_stats[group] = {
                    'mean': mean_val,
                    'sem': sem_val,
                    'data': group_data['median_beta'].values
                }
            else:
                group_stats[group] = {
                    'mean': np.nan,
                    'sem': np.nan,
                    'data': np.array([])
                }
    
    # Setup bar positions
    bar_width = 0.35
    index = np.arange(4)  # 4 brain regions
    
    # Create bars
    short_bars = plt.bar(index - bar_width/2, 
                         [group_stats[f'{area}_short']['mean'] for area in ['RSC', 'PPC', 'M2', 'S1']], 
                         bar_width,
                         yerr=[group_stats[f'{area}_short']['sem'] for area in ['RSC', 'PPC', 'M2', 'S1']],
                         color=[plt_colors[area] for area in ['RSC', 'PPC', 'M2', 'S1']],
                         alpha=0.7,
                         edgecolor='black',
                         linewidth=1,
                         error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1},
                         label='Short Task')
    
    long_bars = plt.bar(index + bar_width/2, 
                        [group_stats[f'{area}_long']['mean'] for area in ['RSC', 'PPC', 'M2', 'S1']], 
                        bar_width,
                        yerr=[group_stats[f'{area}_long']['sem'] for area in ['RSC', 'PPC', 'M2', 'S1']],
                        color=[plt_colors[area] for area in ['RSC', 'PPC', 'M2', 'S1']],
                        alpha=0.3,
                        edgecolor='black',
                        linewidth=1,
                        error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1},
                        label='Long Task')
    
    # Add scatter points for individual sessions
    for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
        # Short condition
        short_data = group_stats[f'{area}_short']['data']
        if len(short_data) > 0:
            short_jitter = np.random.normal(0, 0.03, size=len(short_data))
            plt.scatter(i - bar_width/2 + short_jitter, short_data, color='black', s=15, alpha=0.6)
        
        # Long condition
        long_data = group_stats[f'{area}_long']['data']
        if len(long_data) > 0:
            long_jitter = np.random.normal(0, 0.03, size=len(long_data))
            plt.scatter(i + bar_width/2 + long_jitter, long_data, color='black', s=15, alpha=0.6)
    
    # Add significance markers
    # If mixed_effects_results is provided, use those p-values
    # Otherwise, calculate Mann-Whitney tests
    if mixed_effects_results and 'mixed_effects' in mixed_effects_results:
        # Using the same p-values as in the mixed effects model
        p_values = {}
        mixed_results = mixed_effects_results['mixed_effects']
        
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            if area in mixed_results and 'p_value' in mixed_results[area]:
                p_values[area] = mixed_results[area]['p_value']
            else:
                p_values[area] = 1.0  # No significance if data is missing
                
        # Add significance markers based on mixed effects p-values
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            if area in p_values and p_values[area] < 0.05:
                # Add line connecting the bars
                bar_height = 0.18  # Fixed position for significance bars
                plt.plot([i - bar_width/2, i + bar_width/2], [bar_height, bar_height], 'k-', linewidth=1)
                
                # Add stars
                plt.text(i, 0.19, sig_stars(p_values[area]), ha='center', fontsize=12)
                
        # Add note about statistical approach
        subtitle = "Statistical significance based on mixed effects models (same as Figure6_beta_mixed_effects)"
        plt.figtext(0.5, 0.94, subtitle, ha='center', fontsize=10, fontweight='normal')
    else:
        # Fallback to Mann-Whitney tests if no mixed effects results provided
        p_values = {}
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            # Get short and long data
            short_data = group_stats[f'{area}_short']['data']
            long_data = group_stats[f'{area}_long']['data']
            
            if len(short_data) > 0 and len(long_data) > 0:
                # Perform Mann-Whitney test
                stat, p_val = stats.mannwhitneyu(short_data, long_data, alternative='two-sided')
                p_values[area] = p_val
            else:
                p_values[area] = 1.0  # No significance if data is missing
        
        # Apply FDR correction to p-values
        areas = list(p_values.keys())
        p_vals = [p_values[area] for area in areas]
        if p_vals:
            reject, p_vals_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
            
            # Update p-values with corrected values
            for i, area in enumerate(areas):
                p_values[area] = p_vals_corrected[i]
        
        # Add significance markers based on corrected p-values
        for i, area in enumerate(['RSC', 'PPC', 'M2', 'S1']):
            if area in p_values and p_values[area] < 0.05:
                # Add line connecting the bars
                bar_height = 0.18  # Fixed position for significance bars
                plt.plot([i - bar_width/2, i + bar_width/2], [bar_height, bar_height], 'k-', linewidth=1)
                
                # Add stars
                plt.text(i, 0.19, sig_stars(p_values[area]), ha='center', fontsize=12)
                
        # Add note about statistical approach
        subtitle = "Statistical significance based on Mann-Whitney tests"
        plt.figtext(0.5, 0.94, subtitle, ha='center', fontsize=10, fontweight='normal')
    
    # Customize plot
    plt.xlabel('Brain Region', fontsize=14)
    plt.ylabel('Median |β| value', fontsize=14)
    plt.title(f'Figure {fig_number}: Beta Adaptation by Region (Short vs Long Task)', fontsize=16)
    plt.xticks(index, ['RSC', 'PPC', 'M2', 'S1'], fontsize=12)
    plt.legend(fontsize=12)
    
    # Use linear scale for beta values with fixed limits
    plt.ylim([0, 0.2])
    
    # Remove top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    
    # Add note about statistical significance
    plt.figtext(0.5, 0.01, "* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)", 
               ha='center', fontsize=10, bbox=dict(facecolor='lightyellow', alpha=0.5))
    
    # Save figure
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'Figure{fig_number}_beta_adaptation_barplot.png'), dpi=300)
    plt.savefig(os.path.join(save_dir, f'Figure{fig_number}_beta_adaptation_barplot.svg'))
    plt.close()
    
    print(f"Figure {fig_number} beta adaptation barplot saved.")

params = {
    'main_dir': 'C:/Users/Eva/Desktop/behavior250429/',
    'save_dir': 'C:/Users/Eva/Desktop/Manuscript2025/Figures/2',
    'imaging_dir': 'C:/Users/Eva/Desktop/imaging250429',
    'long_mice': 'long_all_imaging',
    'short_mice': 'short_all_imaging',
    'n_back': 15
}

# Create save directory
os.makedirs(params['save_dir'], exist_ok=True)


def load_session_data(params, task_type='long'):
    """Load session data and organize by brain region.
    
    Parameters:
    -----------
    params : dict
        Dictionary of parameters
    task_type : str
        'long' or 'short' to specify which task data to load
    
    Returns:
    --------
    areas_all : list
        List of lists containing session names for each brain region
    """
    # Select the appropriate mice dataset
    mice_dataset = params['long_mice'] if task_type == 'long' else params['short_mice']
    data_dir = params['main_dir'] + mice_dataset
    
    os.chdir(data_dir)
    list_of_files = glob.glob('*.mat')
    
    # Separate into lists based on brain region
    sessions_RSC = [s[0:12] for s in list_of_files if 'RSC' in s]
    sessions_PPC = [s[0:12] for s in list_of_files if 'PPC' in s]
    sessions_M2 = [s[0:12] for s in list_of_files if 'M2' in s]
    sessions_S1 = [s[0:12] for s in list_of_files if 'S1' in s]
    
    areas_all = [sessions_RSC, sessions_PPC, sessions_M2, sessions_S1]
    
    return areas_all


####################################################################################
##### Process long task data #####
print("\nProcessing LONG task data...")
areas_all_long = load_session_data(params, 'long')

### Process tau values for long task ###
df_long = analyze_tau_values(
    params['imaging_dir'], areas_all_long, params['n_back']
)

# Tau distribution histogram
print("Generating LONG task Figure 2A: Tau distribution histogram")
fig_tau1, ax1 = plot_tau_distribution(
    params['save_dir'], df_long,
    filename_prefix='tau_distribution_long'
)
plt.close(fig_tau1)

# Tau Boxplot comparison
print("Generating LONG task Figure 2B: Boxplot comparison with statistical tests")
fig_tau2, ax2 = plot_tau_boxplot_comparison(
    params['save_dir'], df_long,
    filename_prefix='tau_boxplot_stats_long'
)
plt.close(fig_tau2)

### Process beta values for long task ###
print("Processing LONG task beta values...")
beta_df_long = analyze_beta_values(
    params['imaging_dir'], areas_all_long, params['n_back']
)

# Beta distribution histogram
print("Generating LONG task: Beta distribution histogram")
fig_beta1, ax_beta1 = plot_beta_distribution(
    params['save_dir'], beta_df_long,
    filename_prefix='beta_distribution_long'
)
plt.close(fig_beta1)

# Beta Boxplot comparison
print("Generating LONG task: Beta boxplot comparison with statistical tests")
fig_beta2, ax_beta2 = plot_beta_boxplot_comparison(
    params['save_dir'], beta_df_long,
    filename_prefix='beta_boxplot_stats_long'
)
plt.close(fig_beta2)

##### Process short task data #####
print("\nProcessing SHORT task data...")
areas_all_short = load_session_data(params, 'short')

### Process tau values for short task ###
df_short = analyze_tau_values(
    params['imaging_dir'], areas_all_short, params['n_back']
)

# Tau distribution histogram
print("Generating SHORT task Figure 2A: Tau distribution histogram")
fig_tau3, ax3 = plot_tau_distribution(
    params['save_dir'], df_short,
    filename_prefix='tau_distribution_short'
)
plt.close(fig_tau3)

# Tau boxplot comparison 
## not used in the manuscript
print("Generating SHORT task Figure 2B: Boxplot comparison with statistical tests")
fig_tau4, ax4 = plot_tau_boxplot_comparison(
    params['save_dir'], df_short,
    filename_prefix='tau_boxplot_stats_short'
)
plt.close(fig_tau4)

### Process beta values for short task ###
print("Processing SHORT task beta values...")
beta_df_short = analyze_beta_values(
    params['imaging_dir'], areas_all_short, params['n_back']
)

# Beta distribution histogram
print("Generating SHORT task: Beta distribution histogram")
fig_beta3, ax_beta3 = plot_beta_distribution(
    params['save_dir'], beta_df_short,
    filename_prefix='beta_distribution_short'
)
plt.close(fig_beta3)

# Beta boxplot comparison
## not used in the manuscript
print("Generating SHORT task: Beta boxplot comparison with statistical tests")
fig_beta4, ax_beta4 = plot_beta_boxplot_comparison(
    params['save_dir'], beta_df_short,
    filename_prefix='beta_boxplot_stats_short'
)
plt.close(fig_beta4)

####################################################################################    


####################################################################################    

# Generate Figure 5 - Region adaptation analysis comparing short vs long conditions
print("\nGenerating Figure 5: Region-specific adaptation between short and long tasks...")
adaptation_results, median_df_tau = analyze_region_adaptation(df_short, df_long, params['save_dir'], fig_number=5)

# Add barplot version of Figure 5
plot_tau_adaptation_barplot(median_df_tau, params['save_dir'], fig_number=5, mixed_effects_results=adaptation_results)

# Generate Figure 6 - Beta region adaptation analysis comparing short vs long conditions
print("\nGenerating Figure 6: Beta region-specific adaptation between short and long tasks...")
beta_adaptation_results, median_df_beta = analyze_beta_region_adaptation(beta_df_short, beta_df_long, params['save_dir'], fig_number=6)

# Add barplot version of Figure 6
plot_beta_adaptation_barplot(median_df_beta, params['save_dir'], fig_number=6, mixed_effects_results=beta_adaptation_results)

print("\nAll Figure analyses complete.")
print("Results are saved in:", params['save_dir'])



Processing LONG task data...


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Checking analysis for EZ045 on 230909
Checking analysis for EZ045 on 230915
Checking analysis for EZ050 on 240114
Checking analysis for EZ053 on 240302
Checking analysis for EZ054 on 240101
Checking analysis for EZ054 on 240102
Checking analysis for EZ055 on 240201
Checking analysis for EZ057 on 240228
Checking analysis for MZ108 on 241114
Checking analysis for MZ108 on 241122


  0%|          | 0/8 [00:00<?, ?it/s]

...

Checking analysis for EZ049 on 231122
Checking analysis for EZ049 on 231126
Checking analysis for EZ050 on 231227
Checking analysis for EZ053 on 231227
Checking analysis for EZ054 on 240216
Checking analysis for EZ055 on 240222
Generating SHORT task: Beta distribution histogram
Generating SHORT task: Beta boxplot comparison with statistical tests

Generating Figure 5: Region-specific adaptation between short and long tasks...
Number of sessions per group:
area  condition
M2    long          5
      short         6
PPC   long          8
      short        10
RSC   long         10
      short        13
S1    long          6
      short         6
dtype: int64

Number of mice per group:
area  condition
M2    long         4
      short        5
PPC   long         5
      short        7
RSC   long         7
      short        8
S1    long         4
      short        5
Name: mouse_id, dtype: int64

Median tau values per group:
area  condition
M2    long         2.083669
      short        1.

C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\regression\mixed_linear_model.py:2201: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\regression\mixed_linear_model.py:2201: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization fai

S1 - Mixed effects p-value: 0.2993, effect size: -0.3660

Applying FDR correction to mixed effects p-values:

=== Mixed Effects Model Results with FDR Correction ===
Region | Raw p-value | FDR-corrected p-value | Effect Size | Significance
----------------------------------------------------------------------
RSC   | 0.000000 | 0.000000 | -1.1971 | ***
PPC   | 0.012463 | 0.024925 | -0.5071 | *
M2    | 0.406899 | 0.406899 | -0.1226 | 
S1    | 0.299324 | 0.399098 | -0.3660 | 
Figure 5 tau adaptation barplot saved.

Generating Figure 6: Beta region-specific adaptation between short and long tasks...
Number of sessions per group:
area  condition
M2    long          5
      short         6
PPC   long          8
      short        10
RSC   long         10
      short        13
S1    long          6
      short         6
dtype: int64

Number of mice per group:
area  condition
M2    long         4
      short        5
PPC   long         5
      short        7
RSC   long         7
      short  

C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\regression\mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\regression\mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\regression\mixed_linear_model.py:2201: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
C:\Users\Eva\AppData\Roaming\Python\Python38\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum L

M2 - Mixed effects p-value: 0.7700, effect size: 0.0042
S1 - Mixed effects p-value: 0.5787, effect size: -0.0041

Applying FDR correction to mixed effects p-values:

=== Mixed Effects Model Results with FDR Correction ===
Region | Raw p-value | FDR-corrected p-value | Effect Size | Significance
----------------------------------------------------------------------
RSC   | 0.327526 | 0.655053 | -0.0095 | 
PPC   | 0.128160 | 0.512641 | 0.0134 | 
M2    | 0.770040 | 0.770040 | 0.0042 | 
S1    | 0.578690 | 0.770040 | -0.0041 | 
Figure 6 beta adaptation barplot saved.

All Figure analyses complete.
Results are saved in: C:/Users/Eva/Desktop/Manuscript2025/Figures/2


In [ ]:
# -*- coding: utf-8 -*-
"""
Figure 2 Analysis and Plotting Script

Created on: March 12, 2025
@author: Yu Eva Zhang
"""

# Standard library imports
import os
import glob
import warnings

# Data processing and numerical computation
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'

# Configure warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Import utility functions from utils.py
from utils import (
    plt_color_dir,
    analyze_tau_values,
    plot_tau_distribution,
    plot_tau_boxplot_comparison,
    analyze_region_adaptation,
    analyze_beta_values,
    plot_beta_distribution,
    plot_beta_boxplot_comparison,
    analyze_beta_region_adaptation,
    permutation_test_region_differences,
    permutation_test_beta_region_differences,
    compare_celllevel_taus,
    compare_celllevel_betas,
#     plot_tau_bar_comparison,
#     plot_beta_bar_comparison
)


params = {
    'main_dir': 'C:/Users/Eva/Desktop/behavior250429/',
    'save_dir': 'C:/Users/Eva/Desktop/Manuscript2025/Figures/2',
    'imaging_dir': 'C:/Users/Eva/Desktop/imaging250429',
    'long_mice': 'long_all_imaging',
    'short_mice': 'short_all_imaging',
    'n_back': 15
}

# Create save directory
os.makedirs(params['save_dir'], exist_ok=True)


def load_session_data(params, task_type='long'):
    """Load session data and organize by brain region.
    
    Parameters:
    -----------
    params : dict
        Dictionary of parameters
    task_type : str
        'long' or 'short' to specify which task data to load
    
    Returns:
    --------
    areas_all : list
        List of lists containing session names for each brain region
    """
    # Select the appropriate mice dataset
    mice_dataset = params['long_mice'] if task_type == 'long' else params['short_mice']
    data_dir = params['main_dir'] + mice_dataset
    
    os.chdir(data_dir)
    list_of_files = glob.glob('*.mat')
    
    # Separate into lists based on brain region
    sessions_RSC = [s[0:12] for s in list_of_files if 'RSC' in s]
    sessions_PPC = [s[0:12] for s in list_of_files if 'PPC' in s]
    sessions_M2 = [s[0:12] for s in list_of_files if 'M2' in s]
    sessions_S1 = [s[0:12] for s in list_of_files if 'S1' in s]
    
    areas_all = [sessions_RSC, sessions_PPC, sessions_M2, sessions_S1]
    
    return areas_all


####################################################################################
##### Process long task data #####
print("\nProcessing LONG task data...")
areas_all_long = load_session_data(params, 'long')

### Process tau values for long task ###
df_long = analyze_tau_values(
    params['imaging_dir'], areas_all_long, params['n_back']
)

# Tau distribution histogram
print("Generating LONG task Figure 2A: Tau distribution histogram")
fig_tau1, ax1 = plot_tau_distribution(
    params['save_dir'], df_long,
    filename_prefix='tau_distribution_long'
)
plt.close(fig_tau1)

# Tau Boxplot comparison
print("Generating LONG task Figure 2B: Boxplot comparison with statistical tests")
fig_tau2, ax2 = plot_tau_boxplot_comparison(
    params['save_dir'], df_long,
    filename_prefix='tau_boxplot_stats_long'
)
plt.close(fig_tau2)

### Process beta values for long task ###
print("Processing LONG task beta values...")
beta_df_long = analyze_beta_values(
    params['imaging_dir'], areas_all_long, params['n_back']
)

# Beta distribution histogram
print("Generating LONG task: Beta distribution histogram")
fig_beta1, ax_beta1 = plot_beta_distribution(
    params['save_dir'], beta_df_long,
    filename_prefix='beta_distribution_long'
)
plt.close(fig_beta1)

# Beta Boxplot comparison
print("Generating LONG task: Beta boxplot comparison with statistical tests")
fig_beta2, ax_beta2 = plot_beta_boxplot_comparison(
    params['save_dir'], beta_df_long,
    filename_prefix='beta_boxplot_stats_long'
)
plt.close(fig_beta2)

##### Process short task data #####
print("\nProcessing SHORT task data...")
areas_all_short = load_session_data(params, 'short')

### Process tau values for short task ###
df_short = analyze_tau_values(
    params['imaging_dir'], areas_all_short, params['n_back']
)

# Tau distribution histogram
print("Generating SHORT task Figure 2A: Tau distribution histogram")
fig_tau3, ax3 = plot_tau_distribution(
    params['save_dir'], df_short,
    filename_prefix='tau_distribution_short'
)
plt.close(fig_tau3)

# Tau boxplot comparison 
print("Generating SHORT task Figure 2B: Boxplot comparison with statistical tests")
fig_tau4, ax4 = plot_tau_boxplot_comparison(
    params['save_dir'], df_short,
    filename_prefix='tau_boxplot_stats_short'
)
plt.close(fig_tau4)

### Process beta values for short task ###
print("Processing SHORT task beta values...")
beta_df_short = analyze_beta_values(
    params['imaging_dir'], areas_all_short, params['n_back']
)

# Beta distribution histogram
print("Generating SHORT task: Beta distribution histogram")
fig_beta3, ax_beta3 = plot_beta_distribution(
    params['save_dir'], beta_df_short,
    filename_prefix='beta_distribution_short'
)
plt.close(fig_beta3)

# Beta boxplot comparison
print("Generating SHORT task: Beta boxplot comparison with statistical tests")
fig_beta4, ax_beta4 = plot_beta_boxplot_comparison(
    params['save_dir'], beta_df_short,
    filename_prefix='beta_boxplot_stats_short'
)
plt.close(fig_beta4)
####################################################################################    


####################################################################################    

# Generate Figure 5 - Region adaptation analysis comparing short vs long conditions
print("\nGenerating Figure 5: Region-specific adaptation between short and long tasks...")
adaptation_results, median_df_tau = analyze_region_adaptation(df_short, df_long, params['save_dir'], fig_number=5)

# Generate Figure 6 - Beta region adaptation analysis comparing short vs long conditions
print("\nGenerating Figure 6: Beta region-specific adaptation between short and long tasks...")
beta_adaptation_results = analyze_beta_region_adaptation(beta_df_short, beta_df_long, params['save_dir'], fig_number=6)

#     # Generate Figure 7 - Cell-level tau comparison between short and long tasks
#     print("\nGenerating Figure 7: Cell-level tau comparison between short and long tasks...")
#     celllevel_tau_results = compare_celllevel_taus(df_short, df_long, params['save_dir'], fig_number=7)

#     # Generate Figure 8 - Cell-level beta comparison between short and long tasks
#     print("\nGenerating Figure 8: Cell-level beta comparison between short and long tasks...")
#     celllevel_beta_results = compare_celllevel_betas(beta_df_short, beta_df_long, params['save_dir'], fig_number=8)

####################################################################################


####################################################################################
#     # Run permutation test for region differences with balanced sessions
#     print("\nRunning permutation test for region differences with balanced sessions...")
#     print("This ensures fair comparison by using an equal number of sessions for each brain region")
#     perm_results = permutation_test_region_differences(
#         df_short, df_long, 
#         n_permutations=10000, 
#         balance_sessions=True,
#         n_subsamples=1000,
#         random_seed=42
#     )

#     # Print permutation test results for tau
#     print("\nPermutation test results (tau) with balanced sessions:")
#     for region, result in perm_results.items():
#         print(f"\nRSC vs {region}:")
#         print(f"  Observed difference: {result['observed_diff']:.3f}")
#         print(f"  Raw p-value: {result['raw_p_value']:.4f}")
#         print(f"  FDR-corrected p-value: {result['corrected_p_value']:.4f}")

#         if result['used_balanced_sampling']:
#             print(f"  Used balanced sampling with {result['min_sessions_per_condition']['long']} long sessions " +
#                   f"and {result['min_sessions_per_condition']['short']} short sessions per region")

#         # Print session counts for reference
#         print(f"  Original session counts:")
#         print(f"    RSC: {result['session_counts']['RSC']['long']} long, {result['session_counts']['RSC']['short']} short")
#         print(f"    {region}: {result['session_counts'][region]['long']} long, {result['session_counts'][region]['short']} short")



#     # Run permutation test for beta region differences with balanced sessions
#     print("\nRunning permutation test for beta region differences with balanced sessions...")
#     print("This ensures fair comparison by using an equal number of sessions for each brain region")
#     beta_perm_results = permutation_test_beta_region_differences(
#         beta_df_short, beta_df_long,
#         n_permutations=10000,
#         balance_sessions=True,
#         n_subsamples=1000,
#         random_seed=42
#     )

#     # Print beta permutation test results
#     print("\nPermutation test results (beta) with balanced sessions:")
#     for region, result in beta_perm_results.items():
#         print(f"\nRSC vs {region}:")
#         print(f"  Observed difference: {result['observed_diff']:.3f}")
#         print(f"  Raw p-value: {result['raw_p_value']:.4f}")
#         print(f"  FDR-corrected p-value: {result['corrected_p_value']:.4f}")

#         if result['used_balanced_sampling']:
#             print(f"  Used balanced sampling with {result['min_sessions_per_condition']['long']} long sessions " +
#                   f"and {result['min_sessions_per_condition']['short']} short sessions per region")

#         # Print session counts for reference
#         print(f"  Original session counts:")
#         print(f"    RSC: {result['session_counts']['RSC']['long']} long, {result['session_counts']['RSC']['short']} short")
#         print(f"    {region}: {result['session_counts'][region]['long']} long, {result['session_counts'][region]['short']} short")

#     # Also run without session balancing for comparison
#     print("\nRunning permutation tests WITHOUT session balancing for comparison...")
#     perm_results_unbalanced = permutation_test_region_differences(
#         df_short, df_long, 
#         n_permutations=1000, 
#         balance_sessions=False,
#         random_seed=42
#     )

#     beta_perm_results_unbalanced = permutation_test_beta_region_differences(
#         beta_df_short, beta_df_long,
#         n_permutations=1000,
#         balance_sessions=False,
#         random_seed=42
#     )

#     # Compare results
#     print("\nComparison of p-values with and without session balancing:")
#     print("\nTau values:")
#     for region in perm_results.keys():
#         print(f"\nRSC vs {region}:")
#         print(f"  Balanced:   raw p={perm_results[region]['raw_p_value']:.4f}, corrected p={perm_results[region]['corrected_p_value']:.4f}")
#         print(f"  Unbalanced: raw p={perm_results_unbalanced[region]['raw_p_value']:.4f}, corrected p={perm_results_unbalanced[region]['corrected_p_value']:.4f}")

#     print("\nBeta values:")
#     for region in beta_perm_results.keys():
#         print(f"\nRSC vs {region}:")
#         print(f"  Balanced:   raw p={beta_perm_results[region]['raw_p_value']:.4f}, corrected p={beta_perm_results[region]['corrected_p_value']:.4f}")
#         print(f"  Unbalanced: raw p={beta_perm_results_unbalanced[region]['raw_p_value']:.4f}, corrected p={beta_perm_results_unbalanced[region]['corrected_p_value']:.4f}")

print("\nAll Figure analyses complete.")
print("Results are saved in:", params['save_dir'])


# fraction

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from os.path import join as pjoin
from utils import plt_color_dir
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
from statsmodels.stats.multitest import multipletests

def calculate_mean_fraction_modulated(cell_stats):
    """Calculate mean fraction modulated for each brain area.
    
    Parameters:
    -----------
    cell_stats : dict
        Dictionary containing cell statistics for each session
        
    Returns:
    --------
    dict
        Dictionary with mean fraction modulated for each brain area
    """
    # Initialize dictionaries to store fractions for each area
    area_fractions = {
        'RSC': [],
        'PPC': [],
        'M2': [],
        'S1': []
    }
    
    # Collect fractions for each area
    for session, stats in cell_stats.items():
        area = stats['area']
        if area in area_fractions:
            area_fractions[area].append(stats['fraction_modulated'])
    
    # Calculate means
    mean_fractions = {}
    for area, fractions in area_fractions.items():
        if fractions:  # Only calculate if there are values
            mean_fractions[area] = sum(fractions) / len(fractions)
        else:
            mean_fractions[area] = 0
    
    return mean_fractions

def analyze_cell_counts(save_dir, areas_all, hemi_all, n_back):
    """Analyze cell counts and modulation for each session.
    
    Parameters:
    -----------
    save_dir : str
        Directory where xarray datasets are stored
    areas_all : list
        List of lists containing session names for each brain region
    hemi_all : list
        List of lists containing hemisphere information for each brain region
    n_back : int
        Number of trials back for history analysis
        
    Returns:
    --------
    dict
        Dictionary with session names as keys and cell statistics as values
    """
    # Define area names based on the structure of areas_all
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    # Create dictionary to store cell statistics
    cell_stats = {}
    
    for area_idx in range(len(areas_all)):
        area_name = area_names[area_idx]
        sessions_all = areas_all[area_idx]
        hemispheres = hemi_all[area_idx]
        
        for ss in range(len(sessions_all)):
            session_name = sessions_all[ss]
            hemisphere = hemispheres[ss]
            mouse_id = session_name[:5]  # Extract mouse ID (first 5 characters)
            date = session_name[6:12]
            
            print(f'Analyzing cells for {mouse_id} on {date} ({area_name}, hemisphere: {hemisphere})')
            
            # Load xarray dataset
            try:
                output_xarray = xr.open_dataset(pjoin(save_dir, 'cellfits_halves_compare_mdl', 
                                                     'sse_{}_{}hist_updated.nc'.format(session_name, n_back)))
                
                # Get total number of cells
                total_cells = len(output_xarray.cell)
                
                # Apply criteria for cell selection
                crit_RCp1 = output_xarray.sel(mdl_type='Rp1', half=0).p_beta_RC < 0.05
                crit_sig_half1 = output_xarray.sel(mdl_type='exp_r', half=1).p_beta_RC < 0.05
                crit_sig_half2 = output_xarray.sel(mdl_type='exp_r', half=2).p_beta_RC < 0.05
                
                # Extract data to pandas DataFrame
                out_pd = output_xarray.sel(half=0, mdl_type='exp_r').to_dataframe()[['beta_RC', 'tau_RC']]
                
                # Add criteria columns
                out_pd['crit_RCp1'] = crit_RCp1
                out_pd['crit_sig_half1'] = crit_sig_half1
                out_pd['crit_sig_half2'] = crit_sig_half2
                
                # Filter values based on criteria
                filtered_data = out_pd.loc[(out_pd['crit_sig_half1'] == True) & 
                                (out_pd['crit_sig_half2'] == True) & 
                                (out_pd['tau_RC'] < 99.9) & 
                                (out_pd['tau_RC'] >= 0.11)]
                
                # Determine left vs right preference based on beta_RC sign
                left_preferring = filtered_data[filtered_data['beta_RC'] > 0]
                right_preferring = filtered_data[filtered_data['beta_RC'] < 0]
                
                left_cells = len(left_preferring)
                right_cells = len(right_preferring)
                
                # Calculate fractions
                modulated_cells = len(filtered_data)
                fraction_modulated = modulated_cells / total_cells if total_cells > 0 else 0
                fraction_left = left_cells / modulated_cells if modulated_cells > 0 else 0
                fraction_right = right_cells / modulated_cells if modulated_cells > 0 else 0
                
                # Determine ipsilateral and contralateral fractions based on hemisphere
                if hemisphere == 'l':  # Left hemisphere
                    fraction_ipsi = fraction_left    # Left preferring = ipsilateral
                    fraction_contra = fraction_right # Right preferring = contralateral
                else:  # Right hemisphere (hemisphere == 'r')
                    fraction_ipsi = fraction_right   # Right preferring = ipsilateral
                    fraction_contra = fraction_left  # Left preferring = contralateral
                
                cell_stats[session_name] = {
                    'area': area_name,
                    'mouse_id': mouse_id,
                    'date': date,
                    'hemisphere': hemisphere,
                    'total_cells': total_cells,
                    'modulated_cells': modulated_cells,
                    'fraction_modulated': fraction_modulated,
                    'fraction_left': fraction_left,
                    'fraction_right': fraction_right,
                    'fraction_ipsi': fraction_ipsi,
                    'fraction_contra': fraction_contra
                }
                
            except Exception as e:
                print(f"Error processing session {session_name}: {e}")
                cell_stats[session_name] = {
                    'area': area_name,
                    'mouse_id': mouse_id,
                    'date': date,
                    'hemisphere': hemisphere,
                    'total_cells': 0,
                    'modulated_cells': 0,
                    'fraction_modulated': 0,
                    'fraction_left': 0,
                    'fraction_right': 0,
                    'fraction_ipsi': 0,
                    'fraction_contra': 0
                }
    
    return cell_stats

def load_session_data(params, task_type='long'):
    """Load session data and organize by brain region."""
    # Select the appropriate mice dataset
    mice_dataset = params['long_mice'] if task_type == 'long' else params['short_mice']
    data_dir = params['main_dir'] + mice_dataset
    
    os.chdir(data_dir)
    list_of_files = glob.glob('*.mat')
    
    # Separate into lists based on brain region
    sessions_RSC = [s[0:12] for s in list_of_files if 'RSC' in s]
    sessions_PPC = [s[0:12] for s in list_of_files if 'PPC' in s]
    sessions_M2 = [s[0:12] for s in list_of_files if 'M2' in s]
    sessions_S1 = [s[0:12] for s in list_of_files if 'S1' in s]
    
    # Extract hemisphere information
    hemi_RSC = [s[-5] for s in list_of_files if 'RSC' in s]  # Get last character ('r' or 'l')
    hemi_PPC = [s[-5] for s in list_of_files if 'PPC' in s]
    hemi_M2 = [s[-5] for s in list_of_files if 'M2' in s]
    hemi_S1 = [s[-5] for s in list_of_files if 'S1' in s]
    
    areas_all = [sessions_RSC, sessions_PPC, sessions_M2, sessions_S1]
    hemi_all = [hemi_RSC, hemi_PPC, hemi_M2, hemi_S1]
    
    return areas_all, hemi_all

def plot_fraction_modulated(mean_fractions, cell_stats, task_type, save_dir, me_results=None):
    """Create a bar plot of mean fraction modulated cells by brain region with SEM error bars
    and individual session data points.
    
    Parameters:
    -----------
    mean_fractions : dict
        Dictionary with mean fraction modulated for each brain area
    cell_stats : dict
        Dictionary containing cell statistics for each session
    task_type : str
        'long' or 'short' to specify which task data to plot
    save_dir : str
        Directory to save the plot
    me_results : dict, optional
        Dictionary with mixed effects analysis results
    """
    # Get colors from plt_color_dir
    colors = plt_color_dir()
    
    # Create figure and axis with smaller size
    fig, ax = plt.subplots(figsize=(4, 3))  # Half the original size
    
    # Get regions and their fractions
    regions = list(mean_fractions.keys())
    
    # Calculate SEM for each region
    sem_dict = {}
    for region in regions:
        fractions = [stats['fraction_modulated'] for stats in cell_stats.values() 
                    if stats['area'] == region]
        sem_dict[region] = np.std(fractions) / np.sqrt(len(fractions))
    
    # Create bar plot with error bars
    x_pos = np.arange(len(regions))
    
    # Plot individual session data points first (so they appear below bars)
    for i, region in enumerate(regions):
        # Get all sessions for this region
        region_sessions = [stats for stats in cell_stats.values() 
                         if stats['area'] == region]
        # Add jitter to x positions
        x_jitter = np.random.normal(0, 0.1, len(region_sessions))
        # Plot individual points
        ax.scatter([i + x for x in x_jitter],
                  [s['fraction_modulated'] for s in region_sessions],
                  color='gray', alpha=0.5, s=20,  # Smaller point size
                  zorder=3)  # Ensure points are above bars
    
    # Then plot the bars
    bars = ax.bar(x_pos, [mean_fractions[r] for r in regions], 
                 color=[colors[r] for r in regions],
                 yerr=[sem_dict[r] for r in regions],
                 capsize=5,
                 zorder=2)  # Bars below points
    
    # Add asterisks for significant differences from RSC
    if me_results is not None:
        # Get the index of RSC
        rsc_idx = regions.index('RSC')
        rsc_height = mean_fractions['RSC'] + sem_dict['RSC']
        
        # Add asterisks for each comparison
        for i, region in enumerate(regions):
            if region != 'RSC':
                comparison = f'RSC_vs_{region}'
                if comparison in me_results:
                    p_value = me_results[comparison]['p_value']
                    if p_value < 0.001:
                        symbol = '***'
                    elif p_value < 0.01:
                        symbol = '**'
                    elif p_value < 0.05:
                        symbol = '*'
                    else:
                        symbol = ''
                    
                    if symbol:
                        # Position the asterisk above the higher bar
                        max_height = max(mean_fractions['RSC'], mean_fractions[region])
                        ax.text((rsc_idx + i) / 2, max_height + 0.02, symbol,
                               ha='center', va='bottom', fontsize=8)
    
    # Customize plot
    ax.set_xticks(x_pos)
    ax.set_xticklabels(regions)
    ax.set_ylabel('Fraction of Modulated Cells', fontsize=8)
    ax.set_title(f'Fraction of Modulated Cells by Region - {task_type.upper()} Task', fontsize=9)
    
    # Set consistent y-axis limits and ticks
    ax.set_ylim(0, 0.2)
    ax.set_yticks([0, 0.1, 0.2])
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Adjust tick label sizes
    ax.tick_params(axis='both', which='major', labelsize=8)
    
    # Save plot
    plt.savefig(pjoin(save_dir, f'fraction_modulated_{task_type}.svg'), dpi=300, bbox_inches='tight')
    plt.savefig(pjoin(save_dir, f'fraction_modulated_{task_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def run_mixed_effects_analysis(cell_stats):
    """Run mixed effects model comparing RSC with other brain areas.
    
    Parameters:
    -----------
    cell_stats : dict
        Dictionary containing cell statistics for each session
        
    Returns:
    --------
    dict
        Dictionary with model results for each comparison
    """
    # Convert cell_stats to DataFrame
    data = []
    for session, stats in cell_stats.items():
        data.append({
            'mouse_id': stats['mouse_id'],
            'area': stats['area'],
            'fraction_modulated': stats['fraction_modulated']
        })
    df = pd.DataFrame(data)
    
    # Create dummy variables for area comparisons
    df['is_RSC'] = (df['area'] == 'RSC').astype(int)
    df['is_PPC'] = (df['area'] == 'PPC').astype(int)
    df['is_M2'] = (df['area'] == 'M2').astype(int)
    df['is_S1'] = (df['area'] == 'S1').astype(int)
    
    # Run mixed effects models
    results = {}
    
    # RSC vs PPC
    model_ppc = smf.mixedlm("fraction_modulated ~ is_PPC", 
                           df, 
                           groups=df["mouse_id"]).fit()
    results['RSC_vs_PPC'] = {
        'p_value': model_ppc.pvalues['is_PPC'],
        'coef': model_ppc.params['is_PPC'],
        'ci_lower': model_ppc.conf_int().loc['is_PPC', 0],
        'ci_upper': model_ppc.conf_int().loc['is_PPC', 1]
    }
    
    # RSC vs M2
    model_m2 = smf.mixedlm("fraction_modulated ~ is_M2", 
                          df, 
                          groups=df["mouse_id"]).fit()
    results['RSC_vs_M2'] = {
        'p_value': model_m2.pvalues['is_M2'],
        'coef': model_m2.params['is_M2'],
        'ci_lower': model_m2.conf_int().loc['is_M2', 0],
        'ci_upper': model_m2.conf_int().loc['is_M2', 1]
    }
    
    # RSC vs S1
    model_s1 = smf.mixedlm("fraction_modulated ~ is_S1", 
                          df, 
                          groups=df["mouse_id"]).fit()
    results['RSC_vs_S1'] = {
        'p_value': model_s1.pvalues['is_S1'],
        'coef': model_s1.params['is_S1'],
        'ci_lower': model_s1.conf_int().loc['is_S1', 0],
        'ci_upper': model_s1.conf_int().loc['is_S1', 1]
    }
    
    return results

def print_area_statistics(cell_stats, task_type):
    """Print statistics for each brain area including total neurons, fields, and mice.
    
    Parameters:
    -----------
    cell_stats : dict
        Dictionary containing cell statistics for each session
    task_type : str
        'long' or 'short' to specify which task data
    """
    # Initialize dictionaries to store statistics for each area
    area_stats = {
        'RSC': {'total_neurons': 0, 'fields': 0, 'mice': set()},
        'PPC': {'total_neurons': 0, 'fields': 0, 'mice': set()},
        'M2': {'total_neurons': 0, 'fields': 0, 'mice': set()},
        'S1': {'total_neurons': 0, 'fields': 0, 'mice': set()}
    }
    
    # Collect statistics
    neurons_per_session = []
    for session, stats in cell_stats.items():
        area = stats['area']
        if area in area_stats:
            area_stats[area]['total_neurons'] += stats['total_cells']
            area_stats[area]['fields'] += 1
            area_stats[area]['mice'].add(stats['mouse_id'])
            neurons_per_session.append(stats['total_cells'])
    
    # Print statistics
    print(f"\nStatistics for {task_type.upper()} task:")
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        stats = area_stats[area]
        print(f"{area}: n={stats['total_neurons']} neurons from {stats['fields']} fields in {len(stats['mice'])} mice")
    
    # Print combined statistics across all sessions
    mean_all = np.mean(neurons_per_session)
    std_all = np.std(neurons_per_session)
    print(f"\nAcross all sessions: {mean_all:.1f} ± {std_all:.1f} neurons per session (mean ± SD)")

def plot_laterality_preference(cell_stats, task_type, save_dir, me_results=None):
    """Create a boxplot showing ipsilateral and contralateral preferences by brain region.
    
    Parameters:
    -----------
    cell_stats : dict
        Dictionary containing cell statistics for each session
    task_type : str
        'long' or 'short' to specify which task data to plot
    save_dir : str
        Directory to save the plot
    me_results : dict, optional
        Dictionary with mixed effects analysis results for significance markers
    """
    # Get colors from plt_color_dir
    colors = plt_color_dir()
    
    # Create a DataFrame for the boxplot
    data = []
    for session, stats in cell_stats.items():
        # Add ipsilateral data point
        data.append({
            'Area': stats['area'],
            'Preference': 'Ipsilateral',
            'Fraction': stats['fraction_ipsi'],
            'Session': session
        })
        
        # Add contralateral data point
        data.append({
            'Area': stats['area'],
            'Preference': 'Contralateral',
            'Fraction': stats['fraction_contra'],
            'Session': session
        })
    
    df = pd.DataFrame(data)
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Add a horizontal dashed grey line at y=0.5
    ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.7, zorder=0)
    
    # Areas and preferences for iterating
    areas = ['RSC', 'PPC', 'M2', 'S1']
    preferences = ['Contralateral', 'Ipsilateral']
    
    # Create positions for the boxplots - evenly spaced as requested
    positions = []
    labels = []
    
    for i, area in enumerate(areas):
        # Calculate positions: 1,2 for RSC; 3,4 for PPC; 5,6 for M2; 7,8 for S1
        contra_pos = i * 2 + 1
        ipsi_pos = i * 2 + 2
        
        # Add positions for contra and ipsi
        positions.extend([contra_pos, ipsi_pos])
        
        # Add labels for x-axis
        labels.extend([f"{area}\nContra", f"{area}\nIpsi"])
        
        # Get area color
        area_color = colors[area]
        area_data = df[df['Area'] == area]
        
        # Create boxplots for contralateral and ipsilateral preferences
        for j, pref in enumerate(preferences):
            # Get data for this preference
            pref_data = area_data[area_data['Preference'] == pref]['Fraction']
            
            # Position for this box (1,2 or 3,4 or 5,6 or 7,8)
            box_pos = (i * 2 + 1) + j
            
            # Create boxplot
            bp = ax.boxplot(
                pref_data,
                positions=[box_pos],
                widths=0.7,
                patch_artist=True,
                showfliers=False,
                medianprops={'color': area_color, 'linewidth': 2},
                boxprops={'facecolor': 'none', 'edgecolor': area_color, 'linewidth': 1.5},
                whiskerprops={'color': area_color, 'linewidth': 1.5},
                capprops={'color': area_color, 'linewidth': 1.5}
            )
    
    # Add individual data points
    for i, area in enumerate(areas):
        area_color = colors[area]
        
        for j, pref in enumerate(preferences):
            # Get data for this area and preference
            subset = df[(df['Area'] == area) & (df['Preference'] == pref)]
            points = subset['Fraction']
            
            # Calculate position with jitter
            pos = (i * 2 + 1) + j
            x_jitter = np.random.normal(0, 0.06, size=len(points))
            x_positions = [pos + jit for jit in x_jitter]
            
            # Plot points
            ax.scatter(
                x_positions,
                points,
                color='black',
                alpha=0.6,
                s=20,
                zorder=3
            )
    
    # Add significance markers if ME results are provided
    if me_results is not None:
        for i, area in enumerate(areas):
            comparison = f'{area}_contra_vs_ipsi'
            if comparison in me_results:
                # Use FDR-corrected p-value if available, otherwise use uncorrected p-value
                if 'fdr_corrected_p' in me_results[comparison]:
                    p_value = me_results[comparison]['fdr_corrected_p']
                else:
                    p_value = me_results[comparison]['p_value']
                
                # Determine significance symbol
                if p_value < 0.001:
                    symbol = '***'
                elif p_value < 0.01:
                    symbol = '**'
                elif p_value < 0.05:
                    symbol = '*'
                else:
                    symbol = 'ns'
                
                if symbol != 'ns':
                    # Determine y position for the significance bar
                    area_data = df[df['Area'] == area]
                    max_val = area_data['Fraction'].max() + 0.05
                    
                    # Position for significance bar (between contra and ipsi positions)
                    contra_pos = i * 2 + 1
                    ipsi_pos = i * 2 + 2
                    
                    # Add significance bar and asterisk connecting contra and ipsi
                    ax.plot([contra_pos, ipsi_pos], [max_val, max_val], color='black', linewidth=1)
                    ax.text((contra_pos + ipsi_pos) / 2, max_val + 0.02, symbol, ha='center', va='bottom', fontsize=10)
    
    # Set x-ticks and labels at box positions
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    
    # Add vertical lines between brain areas for visual separation
    for i in range(1, 4):  # Add lines between RSC/PPC, PPC/M2, and M2/S1
        line_pos = i * 2 + 0.5  # Position between areas
        ax.axvline(x=line_pos, color='lightgrey', linestyle='-', alpha=0.5, zorder=0)
    
    # Customize plot
    ax.set_ylabel('Fraction of Modulated Cells', fontsize=10)
    ax.set_title(f'Ipsilateral vs Contralateral Preference - {task_type.upper()} Task', fontsize=11)
    
    # Set consistent y-axis limits
    ax.set_ylim(0, 1.0)
    
    # Set x-axis limits to provide some padding
    ax.set_xlim(0.5, 8.5)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Remove legend as it's replaced by x-axis labels
    
    # Save plot
    plt.tight_layout()
    plt.savefig(pjoin(save_dir, f'laterality_preference_{task_type}.svg'), dpi=300, bbox_inches='tight')
    plt.savefig(pjoin(save_dir, f'laterality_preference_{task_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def run_laterality_mixed_effects_analysis(cell_stats):
    """Run mixed effects model comparing contralateral vs ipsilateral preference within each brain area.
    
    Parameters:
    -----------
    cell_stats : dict
        Dictionary containing cell statistics for each session
        
    Returns:
    --------
    dict
        Dictionary with model results for each comparison
    """
    # Convert cell_stats to a long-format DataFrame for analysis
    data = []
    for session, stats in cell_stats.items():
        # Add row for contralateral preference
        data.append({
            'mouse_id': stats['mouse_id'],
            'area': stats['area'],
            'preference': 'contralateral',
            'fraction': stats['fraction_contra'],
            'session': session
        })
        
        # Add row for ipsilateral preference
        data.append({
            'mouse_id': stats['mouse_id'],
            'area': stats['area'],
            'preference': 'ipsilateral',
            'fraction': stats['fraction_ipsi'],
            'session': session
        })
    
    df = pd.DataFrame(data)
    
    # Create indicator variables for preferences
    df['is_contralateral'] = (df['preference'] == 'contralateral').astype(int)
    
    # Run mixed effects models for each area
    results = {}
    areas = ['RSC', 'PPC', 'M2', 'S1']
    
    for area in areas:
        # Filter data for this area
        area_data = df[df['area'] == area].copy()
        
        if len(area_data) > 0:
            # Run mixed effects model
            try:
                model = smf.mixedlm(
                    "fraction ~ is_contralateral", 
                    area_data, 
                    groups=area_data["mouse_id"]
                ).fit()
                
                results[f'{area}_contra_vs_ipsi'] = {
                    'p_value': model.pvalues['is_contralateral'],
                    'coef': model.params['is_contralateral'],
                    'ci_lower': model.conf_int().loc['is_contralateral', 0],
                    'ci_upper': model.conf_int().loc['is_contralateral', 1]
                }
            except Exception as e:
                print(f"Error in mixed effects model for {area}: {e}")
                results[f'{area}_contra_vs_ipsi'] = {
                    'p_value': np.nan,
                    'coef': np.nan,
                    'ci_lower': np.nan,
                    'ci_upper': np.nan,
                    'error': str(e)
                }
    
    # Apply FDR correction across all comparisons
    if all(not pd.isna(results[comp]['p_value']) for comp in results):
        pvals = [results[comp]['p_value'] for comp in results]
        _, fdr_corrected_pvals, _, _ = multipletests(pvals, method='fdr_bh')
        
        # Add corrected p-values to results
        for i, comp in enumerate(results):
            results[comp]['fdr_corrected_p'] = fdr_corrected_pvals[i]
    
    return results

def main():
    """Main function to analyze cell counts."""
    # Initialize parameters
    params = {
        'main_dir': 'C:/Users/Eva/Desktop/behavior250429/',
        'save_dir': 'C:/Users/Eva/Desktop/Manuscript2025/Figures/2',
        'imaging_dir': 'C:/Users/Eva/Desktop/imaging250429',
        'long_mice': 'long_all_imaging',
        'short_mice': 'short_all_imaging',
        'n_back': 15
    }
    
    # Process long task data
    print("\nProcessing LONG task data...")
    areas_all_long, hemi_all_long = load_session_data(params, 'long')
    long_cell_stats = analyze_cell_counts(params['imaging_dir'], areas_all_long, hemi_all_long, params['n_back'])
    
    # Process short task data
    print("\nProcessing SHORT task data...")
    areas_all_short, hemi_all_short = load_session_data(params, 'short')
    short_cell_stats = analyze_cell_counts(params['imaging_dir'], areas_all_short, hemi_all_short, params['n_back'])
    
    # Combine statistics from both conditions
    combined_stats = {**long_cell_stats, **short_cell_stats}
    
    # Calculate mean and SD across all sessions
    neurons_per_session = [stats['total_cells'] for stats in combined_stats.values()]
    mean_all = np.mean(neurons_per_session)
    std_all = np.std(neurons_per_session)
    
    print(f"\nAcross all sessions (both tasks): {mean_all:.1f} ± {std_all:.1f} neurons per session (mean ± SD)")
    
    # Print statistics for each task
    print("\nStatistics for LONG task:")
    print_area_statistics(long_cell_stats, 'long')
    
    print("\nStatistics for SHORT task:")
    print_area_statistics(short_cell_stats, 'short')
    
    # Calculate mean fractions for long task
    long_mean_fractions = calculate_mean_fraction_modulated(long_cell_stats)
    
    # Print cell statistics for long task
    print("\nCell statistics for LONG task sessions:")
    for session, stats in long_cell_stats.items():
        print(f"\n{session} ({stats['area']}):")
        print(f"  Mouse: {stats['mouse_id']}")
        print(f"  Date: {stats['date']}")
        print(f"  Total cells: {stats['total_cells']}")
        print(f"  Modulated cells: {stats['modulated_cells']}")
        print(f"  Fraction modulated: {stats['fraction_modulated']:.2%}")
    
    # Print mean fractions for long task
    print("\nMean fraction modulated for LONG task by brain area:")
    for area, mean_frac in long_mean_fractions.items():
        print(f"{area}: {mean_frac:.2%}")
    
    # Run mixed effects analysis for long task
    print("\nMixed effects analysis for LONG task:")
    long_results = run_mixed_effects_analysis(long_cell_stats)
    for comparison, result in long_results.items():
        print(f"\n{comparison}:")
        print(f"  Coefficient: {result['coef']:.4f}")
        print(f"  95% CI: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
        print(f"  p-value: {result['p_value']:.4f}")
    
    # Create plot for long task
    plot_fraction_modulated(long_mean_fractions, long_cell_stats, 'long', params['save_dir'], long_results)
    
    # Calculate mean fractions for short task
    short_mean_fractions = calculate_mean_fraction_modulated(short_cell_stats)
    
    # Print cell statistics for short task
    print("\nCell statistics for SHORT task sessions:")
    for session, stats in short_cell_stats.items():
        print(f"\n{session} ({stats['area']}):")
        print(f"  Mouse: {stats['mouse_id']}")
        print(f"  Date: {stats['date']}")
        print(f"  Total cells: {stats['total_cells']}")
        print(f"  Modulated cells: {stats['modulated_cells']}")
        print(f"  Fraction modulated: {stats['fraction_modulated']:.2%}")
    
    # Print mean fractions for short task
    print("\nMean fraction modulated for SHORT task by brain area:")
    for area, mean_frac in short_mean_fractions.items():
        print(f"{area}: {mean_frac:.2%}")
    
    # Run mixed effects analysis for short task
    print("\nMixed effects analysis for SHORT task:")
    short_results = run_mixed_effects_analysis(short_cell_stats)
    for comparison, result in short_results.items():
        print(f"\n{comparison}:")
        print(f"  Coefficient: {result['coef']:.4f}")
        print(f"  95% CI: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
        print(f"  p-value: {result['p_value']:.4f}")
    
    # Create plot for short task
    plot_fraction_modulated(short_mean_fractions, short_cell_stats, 'short', params['save_dir'], short_results)
    
    # Calculate mean fractions for combined data
    combined_mean_fractions = calculate_mean_fraction_modulated(combined_stats)
    
    # Print mean fractions for combined data
    print("\nMean fraction modulated for COMBINED data by brain area:")
    for area, mean_frac in combined_mean_fractions.items():
        print(f"{area}: {mean_frac:.2%}")
    
    # Run mixed effects analysis for combined data
    print("\nMixed effects analysis for COMBINED data:")
    combined_results = run_mixed_effects_analysis(combined_stats)

    # Create plot for combined data
    plot_fraction_modulated(combined_mean_fractions, combined_stats, 'combined', params['save_dir'], combined_results)

    # Run laterality mixed effects analysis
    print("\nRunning laterality mixed effects analysis for LONG task...")
    long_laterality_results = run_laterality_mixed_effects_analysis(long_cell_stats)
    for comparison, result in long_laterality_results.items():
        if 'error' not in result:
            print(f"\n{comparison}:")
            print(f"  Coefficient: {result['coef']:.4f}")
            print(f"  95% CI: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
            print(f"  p-value: {result['p_value']:.4f}")
            if 'fdr_corrected_p' in result:
                print(f"  FDR-corrected p-value: {result['fdr_corrected_p']:.4f}")
        else:
            print(f"\n{comparison}: Error - {result['error']}")
    
    print("\nRunning laterality mixed effects analysis for SHORT task...")
    short_laterality_results = run_laterality_mixed_effects_analysis(short_cell_stats)
    for comparison, result in short_laterality_results.items():
        if 'error' not in result:
            print(f"\n{comparison}:")
            print(f"  Coefficient: {result['coef']:.4f}")
            print(f"  95% CI: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
            print(f"  p-value: {result['p_value']:.4f}")
            if 'fdr_corrected_p' in result:
                print(f"  FDR-corrected p-value: {result['fdr_corrected_p']:.4f}")
        else:
            print(f"\n{comparison}: Error - {result['error']}")
    
    print("\nRunning laterality mixed effects analysis for COMBINED data...")
    combined_laterality_results = run_laterality_mixed_effects_analysis(combined_stats)
    for comparison, result in combined_laterality_results.items():
        if 'error' not in result:
            print(f"\n{comparison}:")
            print(f"  Coefficient: {result['coef']:.4f}")
            print(f"  95% CI: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
            print(f"  p-value: {result['p_value']:.4f}")
            if 'fdr_corrected_p' in result:
                print(f"  FDR-corrected p-value: {result['fdr_corrected_p']:.4f}")
        else:
            print(f"\n{comparison}: Error - {result['error']}")
    
    # Create laterality preference plots with significance markers
    plot_laterality_preference(long_cell_stats, 'long', params['save_dir'], long_laterality_results)
    plot_laterality_preference(short_cell_stats, 'short', params['save_dir'], short_laterality_results)
    plot_laterality_preference(combined_stats, 'combined', params['save_dir'], combined_laterality_results)
    
    # Print laterality preference statistics
    print("\nLaterality preference statistics for LONG task:")
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_sessions = [stats for session, stats in long_cell_stats.items() if stats['area'] == area]
        if area_sessions:
            ipsi_mean = np.mean([stats['fraction_ipsi'] for stats in area_sessions])
            contra_mean = np.mean([stats['fraction_contra'] for stats in area_sessions])
            ipsi_sem = np.std([stats['fraction_ipsi'] for stats in area_sessions]) / np.sqrt(len(area_sessions))
            contra_sem = np.std([stats['fraction_contra'] for stats in area_sessions]) / np.sqrt(len(area_sessions))
            print(f"{area}: Ipsilateral: {ipsi_mean:.2%} ± {ipsi_sem:.2%}, Contralateral: {contra_mean:.2%} ± {contra_sem:.2%}")
    
    print("\nLaterality preference statistics for SHORT task:")
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_sessions = [stats for session, stats in short_cell_stats.items() if stats['area'] == area]
        if area_sessions:
            ipsi_mean = np.mean([stats['fraction_ipsi'] for stats in area_sessions])
            contra_mean = np.mean([stats['fraction_contra'] for stats in area_sessions])
            ipsi_sem = np.std([stats['fraction_ipsi'] for stats in area_sessions]) / np.sqrt(len(area_sessions))
            contra_sem = np.std([stats['fraction_contra'] for stats in area_sessions]) / np.sqrt(len(area_sessions))
            print(f"{area}: Ipsilateral: {ipsi_mean:.2%} ± {ipsi_sem:.2%}, Contralateral: {contra_mean:.2%} ± {contra_sem:.2%}")
    
    print("\nLaterality preference statistics for COMBINED data:")
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_sessions = [stats for session, stats in combined_stats.items() if stats['area'] == area]
        if area_sessions:
            ipsi_mean = np.mean([stats['fraction_ipsi'] for stats in area_sessions])
            contra_mean = np.mean([stats['fraction_contra'] for stats in area_sessions])
            ipsi_sem = np.std([stats['fraction_ipsi'] for stats in area_sessions]) / np.sqrt(len(area_sessions))
            contra_sem = np.std([stats['fraction_contra'] for stats in area_sessions]) / np.sqrt(len(area_sessions))
            print(f"{area}: Ipsilateral: {ipsi_mean:.2%} ± {ipsi_sem:.2%}, Contralateral: {contra_mean:.2%} ± {contra_sem:.2%}")

if __name__ == "__main__":
    main() 